# Tuấn — Phase 2: DBNet + MonkeyOCRv2-AS trên TextVQA clean/degraded

**Mục tiêu:** đo thay đổi detection giữa ảnh clean và degraded tương ứng, chưa fine-tune.
Giữ checkpoint, backbone, source revisions và model config đã qua Phase 1.

**Kaggle: attach đủ ba dataset này, bật Internet và GPU T4 ×2:**
- [Clean TextVQA train](https://www.kaggle.com/datasets/anhthu128/text-vqa-train)
- [Degraded 10K do Tuấn tạo](https://www.kaggle.com/datasets/anhthu128/textvqa-suy-thoi-10k-nh)
- [TextOCR v0.1 train GT](https://www.kaggle.com/datasets/anhthu128/textocr-0-1-train-json)

Khác Phase 1: **không tự tải ba bộ dữ liệu này**; phải attach chúng. Không tạo lại degradation.
Cell audit tự tìm đường dẫn thật, xác minh metadata/hashes và xử lý homography trước inference.

**Chạy một lượt:** giữ `RUN_STAGE='FULL'` rồi Run All. Notebook tự chạy audit, lưu GT overlays,
smoke 30 source pairs và tiếp tục full nếu các kiểm tra đạt. Không phải chép hash hoặc xin xác nhận
giữa chừng. Khi mapping/hash/geometry/checkpoint/inference lỗi, notebook vẫn dừng để tránh chấm sai.
Tùy chọn `RUN_STAGE='AUDIT_SMOKE'` chỉ dùng khi muốn dừng ở lượt thử.
Manifest/protocol được snapshot tự động trước inference, kiểm tra lại trước full và giữ nguyên để gửi Paddle.
Nếu chuyển session cần giữ cùng manifest bytes và chính xác ba dataset versions.

Đánh giá tối đa gần 10K clean + 10K degraded; giao hợp lệ có thể nhỏ hơn. Clean chỉ infer một lần/source.
Hai T4 chạy hai subprocess độc lập chia ảnh, không chia đôi model. H200 dùng cùng checkpoint/config,
nhưng cần thay paths cho Linux host, cài đúng stack và đo lại smoke; không có benchmark H200 ở đây.

**Tình trạng khi bàn giao:** kiểm thử CPU với dữ liệu giả lập nhỏ; chưa chạy Phase 2 trên Kaggle/GPU.
Phase 1 Monkey đã đạt evaluation reproduction; kết quả của Paddle phải được người phụ trách xác nhận riêng.
Không có metrics robustness trước khi Tuấn chạy trên dữ liệu thật.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shutil, platform, re, time, textwrap
from datetime import datetime, timezone

ROOT = Path('/kaggle/working/monkey_phase2')
RUN_ID = 'tuan_textvqa_baseline_v1'
RUN_STAGE = 'FULL'  # Run All: audit -> smoke -> full khi kiểm tra đạt. Có thể chọn AUDIT_SMOKE.
EXPECTED_SHARED_MANIFEST_SHA256 = ''  # Tùy chọn: pin hash khi dùng lại manifest đã có; không cần điền lần đầu.
EXPECTED_SHARED_PROTOCOL_SHA256 = ''  # Tùy chọn tương tự; không phải bước phê duyệt thủ công.
GPU_IDS = 'auto'  # auto = tối đa hai GPU, nếu chỉ có H200 đơn thì dùng GPU 0.
SMOKE_SOURCES = 30
MAX_BUILD_JOBS = 2
CUDA_HOME_OVERRIDE = ''
ALLOW_CUDA_MINOR_MISMATCH = False
BOOTSTRAP_REPETITIONS = 1000

assert RUN_STAGE in {'AUDIT_SMOKE','FULL'}
assert re.fullmatch(r'[A-Za-z0-9_-]+',RUN_ID)
ART = ROOT/'artifacts'/RUN_ID
for sub in ('logs','source','scripts'):(ART/sub).mkdir(parents=True,exist_ok=True)
REPO=ROOT/'MonkeyOCRv2';ADDON=REPO/'detection/mmocr';MMOCR_ROOT=ADDON/'mmocr'
VENV=ROOT/'env_py311';PY=VENV/'bin/python'
MONKEY_URL='https://github.com/Yuliang-Liu/MonkeyOCRv2.git'
DETECTOR_REPO='HB16888/MonkeyOCRv2_det';BACKBONE_REPO='zenosai/MonkeyOCRv2-AS'
DETECTOR_FILENAME='dbnet_mkv2vitae_icdar2015.pth'
CONFIG_REL='configs/textdet/dbnet/dbnet_mkv2vitae_1200e_icdar2015_2gpu_adamw.py'
EXPECTED_SOURCE_LOCK={'monkey_commit': '6cc0c0bb4864f2541fa6bc3a8f976cffaac8f359', 'mmcv_commit': 'fc038a386a42d1f36d9ec49169d5817a52c70b8e', 'mmocr_commit': '1dcd6fa6958de22bcb997319833f0ac19c180ec7', 'detector_hf_revision': '58c8bd712a1757a1d21b729ab69910573ecc325b', 'backbone_hf_revision': '51778e7738fdad36ed72288881071b4dcd04b127'}
EXPECTED_CHECKPOINT_SHA256='4b13e26ad5c2e1b13fef45ac608298254d083062d2b9cafba706064bd8e2cf82'
EXPECTED_CONFIG_SHA256='db34e11dff43c8fc0d07bc5807a4e22442d218ccd43183d8b610e1d330be0e2c'

MONKEY_REF=EXPECTED_SOURCE_LOCK['monkey_commit']
lock_path=ART/'source_lock.json'
if lock_path.exists():
    assert json.loads(lock_path.read_text())==EXPECTED_SOURCE_LOCK,'Different source lock: do not mix Phase 1/2 models.'
else:lock_path.write_text(json.dumps(EXPECTED_SOURCE_LOCK,indent=2))
print('Stage:',RUN_STAGE,'| artifacts:',ART)


In [ ]:
import sys, subprocess, importlib.util
from pathlib import Path
shared_deps=Path('/kaggle/working/phase2_shared_dependencies')
missing=[pkg for module,pkg in [('numpy','numpy'),('PIL','Pillow'),('shapely','shapely')]
         if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--target',str(shared_deps),*missing],check=True)
    sys.path.insert(0,str(shared_deps))
print('Shared cell dependencies available; no Paddle/PyTorch required for the data audit.')


## CELL CÔNG KHAI — một manifest cho Monkey và Paddle

Cell tiếp theo chứa **toàn bộ code** `shared_manifest.py`, không import detector và không đọc predictions.
Nó lấy giao theo ID gốc: TextVQA clean ∩ degraded MAIN hiện có ∩ TextOCR train.
Không dùng số câu hỏi làm số ảnh, không tạo ảnh và không suy ID degraded từ tên file.

Code hỗ trợ schema của bốn notebook generator Tuấn đã cung cấp. Nếu file/schema khác hoặc thiếu,
cell dừng và liệt kê file thật; chưa kiểm tra được nội dung ba dataset từ máy soạn notebook.
Mặc định lấy MAIN, không trộn calibration/ablation. Số cuối cùng phải đọc từ audit, không mặc định 10K.

**Protocol v1 được khóa trước khi xem predictions:** ignored transcript `.`/`###`/rỗng;
polygon IoU ≥0,5, maximum-cardinality matching; word coverage ≥0,5; không score sweep.
Model giữ nguyên postprocessing riêng. Word-vs-line và coverage bị box lớn chi phối phải được diễn giải.

Perspective dùng corners đã lưu để dựng H. Chỉ đo trong phần nội dung nguồn còn nhìn thấy ở tất cả
variants của source; biên phản chiếu bị mask. GT có <80% diện tích còn nhìn thấy được ignore ở cả
clean/degraded; các valid annotation IDs giống nhau. Mọi lọc/repair/loại đều có số đếm trong audit.
Đây là lựa chọn protocol của nhóm, **không phải evaluator chính thức ICDAR2015**.

**Bên Paddle nhận cùng bytes `shared_manifest_v1.jsonl` + protocol + SHA256 + evaluator**.
Không chạy một bộ join độc lập rồi mặc định tương đương. `dataset_locations.json` là đường dẫn local,
có thể đổi mount root mà không thay manifest.


In [ ]:
"""Public, detector-independent intersection cell. No model, training or degradation generation.

Supported input metadata schema: the four v2_10k generator notebooks supplied by Tuan.
All paths are discovered from real files; source IDs come from JSON/CSV, never degraded filenames.
The frozen manifest is root-independent. Copy the resulting bytes to Paddle; do not rebuild it.
"""
import csv
import hashlib
import io
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

import numpy as np
from PIL import Image, ImageDraw
from shapely.geometry import Polygon, box, mapping, shape
try:
    from shapely import transform as array_geometry_transform
except ImportError:
    array_geometry_transform = None
from shapely.ops import transform as legacy_geometry_transform

SCHEMA_VERSION = 'textdet-paired-v1'
GENERATOR_PROTOCOL = 'TEXTVQA_REALISTIC_DEGRADATION_V2_10K'
KNOWN_OPERATIONS = {'motion_camera_shake', 'defocus_blur', 'lowlight_sensor_noise',
                    'overexposure_glare', 'low_resolution', 'perspective', 'jpeg_compression'}
DEFAULT_SHARED_CONFIG = dict(
    input_root='/kaggle/input', output_root='/kaggle/working/textdet_shared_v1',
    clean_json='', textocr_json='', main_image_manifest='',
    clean_images_root='', degraded_root='',
    ignore_transcripts=['.', '###', ''], min_word_visibility=0.80,
    min_prediction_roi_fraction=0.50, ignore_prediction_overlap=0.50,
    match_iou=0.50, coverage_threshold=0.50, overlay_sources=12,
)


def canonical_bytes(obj):
    return (json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(',', ':'),
                       allow_nan=False) + '\n').encode('utf-8')


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for data in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(data)
    return digest.hexdigest()


def write_locked(path, data):
    path = Path(path)
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen artifact changed: {path}. Use a new output revision; do not overwrite.')
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    return hashlib.sha256(data).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))


def describe_textocr_schema(raw):
    """Small diagnostics, not the entire large annotation JSON or transcripts."""
    report = dict(root_type=type(raw).__name__)
    if isinstance(raw, dict):
        report['top_level_keys'] = sorted(raw.keys())
        fields = {}
        for key, value in raw.items():
            field = dict(type=type(value).__name__)
            if isinstance(value, (dict, list)):
                field['length'] = len(value)
                example = next(iter(value.values()), None) if isinstance(value, dict) else next(iter(value), None)
                if isinstance(example, dict):
                    field['record_keys'] = sorted(example.keys())
                elif example is not None:
                    field['element_type'] = type(example).__name__
            fields[key] = field
        report['fields'] = fields
    return report


def normalize_textocr(raw):
    """Canonicalize explicit IDs and rebuild the redundant image->annotations index.

    The source of the mapping is each annotation's image_id, NOT filenames, list positions,
    or a guessed association. Polygon points remain unchanged. Existing usable indexes
    are checked against these foreign keys before their order is retained.
    """
    report = describe_textocr_schema(raw)
    if not isinstance(raw, dict):
        raise ValueError(f'TextOCR must be an object: {report}')
    def choose(primary, alias):
        if primary in raw:
            return primary, raw[primary]
        if alias in raw:
            return alias, raw[alias]
        raise ValueError(f'Missing {primary}/{alias}; actual keys={report.get("top_level_keys")}')
    image_key, image_records = choose('imgs', 'images')
    annotation_key, annotation_records = choose('anns', 'annotations')
    def canonical_id(value, label):
        if isinstance(value, bool) or not isinstance(value, (str, int)) or value == '':
            raise ValueError(f'{label}: missing/non-scalar explicit ID: {value!r}')
        return str(value)  # Do not strip leading zeroes, change case, or parse filename stems.
    def index_records(records, label):
        if not isinstance(records, (dict, list)):
            raise ValueError(f'{label} must be dict/list; actual type={type(records).__name__}')
        result = {}
        iterator = records.items() if isinstance(records, dict) else ((None, r) for r in records)
        for key, record in iterator:
            if not isinstance(record, dict):
                raise ValueError(f'{label}: expected object records, got {type(record).__name__}')
            item_id = canonical_id(record.get('id') if key is None else key, label)
            if 'id' in record and canonical_id(record['id'], label) != item_id:
                raise ValueError(f'{label}: dictionary key disagrees with record.id for {item_id}')
            if item_id in result:
                raise ValueError(f'{label}: duplicate explicit ID {item_id}')
            result[item_id] = record
        return result
    images = index_records(image_records, image_key)
    annotations = index_records(annotation_records, annotation_key)
    if not images or not annotations:
        raise ValueError('TextOCR images/annotations are empty; test metadata without GT is not usable here')
    rebuilt = defaultdict(list)
    for ann_id, ann in annotations.items():
        image_id = canonical_id(ann.get('image_id'), f'annotation {ann_id}.image_id')
        if image_id not in images:
            raise ValueError(f'Annotation {ann_id} references unknown image_id {image_id}')
        if 'points' not in ann:
            raise ValueError(f'Annotation {ann_id} has no polygon points; actual record keys={sorted(ann)}. '
                             'Do not substitute bbox or a recognition/VQA annotation as polygon GT.')
        rebuilt[image_id].append(ann_id)
    provided_keys = [key for key in ('img2Anns', 'imgToAnns') if key in raw]
    selected_index, validated_keys = None, []
    unusable_index_types = {}
    for key in provided_keys:
        supplied = raw[key]
        if not isinstance(supplied, dict):
            # This is only a redundant index; explicit annotation foreign keys remain authoritative.
            unusable_index_types[key] = type(supplied).__name__
            continue
        normalized = {}
        for image_id, ids in supplied.items():
            image_id = canonical_id(image_id, key)
            if image_id not in images or not isinstance(ids, list):
                raise ValueError(f'{key}: unknown image ID or non-list annotation IDs for {image_id}')
            ann_ids = [canonical_id(item, key) for item in ids]
            if len(set(ann_ids)) != len(ann_ids):
                raise ValueError(f'{key}: duplicate annotation references for {image_id}')
            if set(ann_ids) != set(rebuilt.get(image_id, [])):
                raise ValueError(f'{key}: links disagree with annotations.image_id for {image_id}; do not guess mapping')
            normalized[image_id] = ann_ids
        if any(set(normalized.get(image_id, [])) != set(ids) for image_id, ids in rebuilt.items()):
            raise ValueError(f'{key}: index omits annotations present in anns; inconsistent GT needs audit')
        if selected_index is None:
            selected_index = normalized
        validated_keys.append(key)
    if selected_index is None:
        selected_index = {image_id: sorted(ids) for image_id, ids in rebuilt.items()}
    report.update(image_field=image_key, annotation_field=annotation_key,
                  canonical_images=len(images), canonical_annotations=len(annotations),
                  validated_index_keys=validated_keys, non_dictionary_index_types=unusable_index_types,
                  mapping_source=('validated ' + validated_keys[0]) if validated_keys else 'rebuilt from anns[*].image_id',
                  polygon_source='unchanged points; no bbox/filename fallback')
    return dict(imgs=images, anns=annotations, img2Anns=selected_index), report


def csv_rows(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))


def one_file(explicit, candidates, label):
    paths = [Path(explicit)] if explicit else sorted(set(Path(p) for p in candidates))
    if len(paths) != 1 or not paths[0].is_file():
        raise RuntimeError(f'{label}: expected one real file; candidates={[str(p) for p in paths]}. '
                           'Set the corresponding SHARED_CONFIG path explicitly. Do not guess.')
    return paths[0].resolve()


def discover(config):
    root = Path(config['input_root']).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f'Attach the three Kaggle datasets; missing {root}')
    # Names are defined by the supplied generator and the original annotation releases.
    clean = one_file(config['clean_json'], root.rglob('TextVQA_0.5.1_train.json'), 'clean_json')
    textocr = one_file(config['textocr_json'], root.rglob('TextOCR_0.1_train.json'), 'textocr_json')
    main_candidates = [p for p in root.rglob('image_manifest.csv') if p.parent.name == 'main']
    main = one_file(config['main_image_manifest'], main_candidates, 'main_image_manifest')
    return dict(clean_json=clean, textocr_json=textocr, main_manifest=main,
                clean_root=Path(config['clean_images_root']).resolve() if config['clean_images_root'] else clean.parent,
                degraded_root=Path(config['degraded_root']).resolve() if config['degraded_root'] else main.parent.parent)


def image_index(root):
    result = defaultdict(list)
    for p in sorted(Path(root).rglob('*')):
        if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}:
            result[p.name].append(p.resolve())
    return result


def resolve_recorded_path(recorded, root, index, expected_hash, kind):
    """Rebind a recorded path by exact suffix/basename and mandatory content hash.

    This does NOT infer image IDs from filenames. ID -> recorded file is in generator metadata.
    """
    if not recorded or not expected_hash or len(expected_hash) != 64:
        raise ValueError(f'{kind}: missing recorded path or SHA256')
    root = Path(root).resolve()
    parts = PurePosixPath(str(recorded).replace('\\', '/')).parts
    candidates = []
    direct = Path(recorded)
    if direct.is_file() and direct.resolve().is_relative_to(root):
        candidates.append(direct.resolve())
    if kind == 'degraded':
        anchors = [i for i in range(len(parts) - 1) if parts[i:i + 2] == ('main', 'images')]
        if len(anchors) == 1:
            rebound = root.joinpath(*parts[anchors[0]:])
            if rebound.is_file():
                candidates.append(rebound.resolve())
    # A unique exact basename is allowed only with the metadata hash check below.
    if not candidates:
        candidates = index.get(parts[-1], []) if parts else []
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise ValueError(f'{kind}: ambiguous/missing file for metadata path {recorded}: {candidates}')
    p = candidates[0]
    if not p.is_relative_to(root) or file_sha256(p) != expected_hash:
        raise ValueError(f'{kind}: bytes/path do not match generator metadata: {p}')
    return p


def image_size(path):
    with Image.open(path) as im:
        if im.getexif().get(274, 1) not in (None, 1):
            raise ValueError('Non-identity EXIF orientation; coordinate frame needs explicit audit')
        size = im.size
        im.convert('RGB').load()  # Decode every paired image; detect damaged files before inference.
    return size


def homography_from_corners(width, height, corners):
    src = np.array([[0., 0.], [width - 1., 0.], [width - 1., height - 1.], [0., height - 1.]])
    dst = np.asarray(corners, dtype=float)
    if dst.shape != (4, 2) or not np.isfinite(dst).all():
        raise ValueError('Perspective corners must be finite 4x2, in generator source-corner order')
    a, b = [], []
    for (x, y), (u, v) in zip(src, dst):
        a.extend([[x, y, 1, 0, 0, 0, -u*x, -u*y], [0, 0, 0, x, y, 1, -v*x, -v*y]])
        b.extend([u, v])
    h = np.append(np.linalg.solve(np.array(a), np.array(b)), 1).reshape(3, 3)
    if not np.isfinite(h).all() or np.linalg.cond(h) > 1e12:
        raise ValueError('Unstable homography')
    projected = project_points(src, h)
    if np.max(np.abs(projected - dst)) > 1e-4:
        raise ValueError('Corner reprojection check failed')
    return h


def project_points(points, h):
    points = np.asarray(points, dtype=float).reshape(-1, 2)
    q = np.column_stack([points, np.ones(len(points))]) @ np.asarray(h).T
    if not np.isfinite(q).all() or np.any(np.abs(q[:, 2]) < 1e-10):
        raise ValueError('Homography crosses projective horizon')
    return q[:, :2] / q[:, 2, None]


def warp_geometry(geom, h):
    h = np.asarray(h, dtype=float)
    def project(x, y, z=None):
        xa, ya = np.asarray(x), np.asarray(y)
        den = h[2, 0]*xa + h[2, 1]*ya + h[2, 2]
        if np.any(np.abs(den) < 1e-10):
            raise ValueError('Projective horizon intersects polygon')
        return ((h[0, 0]*xa + h[0, 1]*ya + h[0, 2])/den,
                (h[1, 0]*xa + h[1, 1]*ya + h[1, 2])/den)
    out = (array_geometry_transform(geom, lambda coordinates: project_points(coordinates, h))
           if array_geometry_transform is not None else legacy_geometry_transform(project, geom))
    if not out.is_valid:
        out = out.buffer(0)
    if out.is_empty or out.area <= 0:
        raise ValueError('Empty warped geometry')
    return out


def source_geometry(record, size, output_size):
    if size != output_size:
        raise ValueError('Generator expects unchanged canvas. Unknown resize/crop; metadata audit needed.')
    params = record['params']
    if isinstance(params, str):
        params = json.loads(params)
    if not isinstance(params, list) or not params:
        raise ValueError('Missing per-image operation metadata')
    ops = [p['operation'] for p in params]
    if not set(ops) <= KNOWN_OPERATIONS:
        raise ValueError(f'Unsupported geometric/unknown operations: {set(ops)-KNOWN_OPERATIONS}')
    perspectives = [p for p in params if p['operation'] == 'perspective']
    if len(perspectives) > 1:
        raise ValueError('Multiple perspective transforms require explicit composition')
    w, h = size
    matrix = homography_from_corners(w, h, perspectives[0]['corners']) if perspectives else np.eye(3)
    canvas = box(0, 0, w, h)
    # Reflected border pixels are outside the source field of view, and must not create GT/FPs.
    source_visible = canvas.intersection(warp_geometry(canvas, np.linalg.inv(matrix)))
    if source_visible.is_empty or source_visible.area / canvas.area < 0.50:
        raise ValueError('Implausible visible field of view')
    return matrix, source_visible, ops


def annotation_geometry(annotation):
    arr = np.asarray(annotation['points'], dtype=float)
    if arr.size < 6 or arr.size % 2 or not np.isfinite(arr).all():
        raise ValueError('Invalid TextOCR polygon coordinates')
    geom = Polygon(arr.reshape(-1, 2))
    repaired = not geom.is_valid
    if repaired:
        geom = geom.buffer(0)
    if geom.is_empty or geom.area <= 0 or geom.geom_type not in {'Polygon', 'MultiPolygon'}:
        raise ValueError('Unusable TextOCR polygon')
    return geom, repaired


def gt_for_source(image_id, textocr, roi, config, audit_counts):
    valid, ignored, removed_ids = [], [], []
    for ann_id in textocr['img2Anns'].get(image_id, []):
        ann = textocr['anns'][str(ann_id)]
        if str(ann.get('image_id', image_id)) != image_id:
            raise ValueError('TextOCR img2Anns / anns image_id mismatch')
        try:
            geom, repaired = annotation_geometry(ann)
        except (ValueError, KeyError):
            audit_counts['unusable_gt_annotations'] += 1
            removed_ids.append(str(ann_id))
            continue
        audit_counts['repaired_gt_annotations'] += int(repaired)
        clipped = geom.intersection(roi)
        visibility = clipped.area / geom.area
        if clipped.is_empty or clipped.area <= 0:
            audit_counts['gt_outside_common_field_of_view'] += 1
            removed_ids.append(str(ann_id))
            continue
        unreadable = str(ann.get('utf8_string', '')).strip() in config['ignore_transcripts']
        explicit_ignore = ann.get('ignore', False) is True or ann.get('ignored', False) is True
        partial = visibility < config['min_word_visibility']
        entry = dict(ann_id=str(ann_id), geometry=mapping(clipped),
                     original_height=float(geom.bounds[3] - geom.bounds[1]), visibility=float(visibility))
        if unreadable or explicit_ignore or partial:
            entry['ignore_reason'] = 'unreadable' if unreadable or explicit_ignore else 'cropped_below_visibility'
            ignored.append(entry)
            audit_counts['gt_ignored_' + entry['ignore_reason']] += 1
        else:
            valid.append(entry)
    return dict(valid=valid, ignored=ignored), removed_ids


def overlay(geometry, drawing, color, width=2):
    if geometry.geom_type == 'Polygon':
        drawing.line(list(geometry.exterior.coords), fill=color, width=width)
    elif hasattr(geometry, 'geoms'):
        for part in geometry.geoms:
            overlay(part, drawing, color, width)


def save_gt_overlays(rows, locations, output, limit):
    output.mkdir(parents=True, exist_ok=True)
    chosen = sorted(rows, key=lambda r: (not r['has_perspective'], r['source_image_id'], r['variant_id']))[:limit]
    for i, row in enumerate(chosen):
        for cond in ('clean', 'degraded'):
            p = Path(locations[cond + '_root']) / row[cond + '_relpath']
            with Image.open(p) as im:
                im = im.convert('RGB')
                draw = ImageDraw.Draw(im)
                overlay(shape(row['roi_' + cond]), draw, '#00BFFF', 3)
                for item in row['gt_' + cond]['ignored']:
                    overlay(shape(item['geometry']), draw, '#AAAAAA')
                for item in row['gt_' + cond]['valid']:
                    overlay(shape(item['geometry']), draw, '#00FF00')
                im.thumbnail((1280, 1280))
                im.save(output / f'{i:02d}_{row["source_image_id"]}_{cond}.jpg', quality=88)


def build_shared_manifest(config=None):
    config = dict(DEFAULT_SHARED_CONFIG, **(config or {}))
    output = Path(config['output_root']).resolve()
    output.mkdir(parents=True, exist_ok=True)
    assert 0 < config['min_word_visibility'] <= 1
    paths = discover(config)
    print('DISCOVERED (real files):', json.dumps({k: str(v) for k, v in paths.items()}, indent=2), flush=True)
    records = csv_rows(paths['main_manifest'])
    required = {'image_id', 'condition', 'level', 'params', 'output_path', 'source_image_path',
                'source_image_sha256', 'output_sha256', 'config_sha256', 'source_manifest_sha256'}
    if not records or not required <= records[0].keys():
        raise ValueError(f'Unsupported generator CSV schema; required={sorted(required)}')
    clean_json = read_json(paths['clean_json'])
    if not isinstance(clean_json.get('data'), list):
        raise ValueError('Unsupported TextVQA schema: expected data list')
    clean_ids = {str(r['image_id']) for r in clean_json['data']}
    del clean_json
    textocr_raw = read_json(paths['textocr_json'])
    schema_path = output / 'textocr_schema.json'
    schema_path.write_bytes(canonical_bytes(describe_textocr_schema(textocr_raw)))
    print('TEXTOCR RAW SCHEMA:', json.dumps(describe_textocr_schema(textocr_raw), indent=2), flush=True)
    try:
        textocr, textocr_schema = normalize_textocr(textocr_raw)
    except ValueError as exc:
        raise ValueError(f'{exc}. Schema diagnostics: {schema_path}') from exc
    schema_path.write_bytes(canonical_bytes(textocr_schema))
    del textocr_raw
    print('TEXTOCR MAPPING:', textocr_schema['mapping_source'], flush=True)
    print('SCHEMAS:', dict(main_columns=sorted(records[0]), main_records=len(records),
                          textvqa_unique_ids=len(clean_ids), textocr_images=len(textocr['imgs'])), flush=True)

    # Verify the actual frozen config against every selected generator record.
    frozen_path = paths['degraded_root'] / 'config/frozen_degradation_config.json'
    policy_path = paths['degraded_root'] / 'metadata/split_policy.json'
    eval_path = paths['degraded_root'] / 'metadata/locked_eval_manifest.csv'
    if not all(p.is_file() for p in (frozen_path, policy_path, eval_path)):
        raise ValueError('Missing frozen config / split policy / locked_eval_manifest. Inspect input; do not guess.')
    frozen, policy = read_json(frozen_path), read_json(policy_path)
    if frozen.get('protocol') != GENERATOR_PROTOCOL or policy.get('protocol') != GENERATOR_PROTOCOL:
        raise ValueError('Not the supplied v2_10k generator protocol; needs an explicit schema adapter')
    frozen_hash, selection_hash = file_sha256(frozen_path), file_sha256(eval_path)
    if frozen.get('eval_manifest_sha256') != selection_hash:
        raise ValueError('Frozen config / selection manifest hash mismatch')
    if frozen.get('train_annotation_sha256') != file_sha256(paths['clean_json']):
        raise ValueError('Attached clean TextVQA JSON is not the source used by the generator')
    selected_rows = csv_rows(eval_path)
    if not selected_rows or '_source_image_sha256' not in selected_rows[0]:
        raise ValueError('Locked selection lacks source-image hashes; cannot verify ID-to-clean mapping')
    selected_ids = {r['_image_id'] for r in selected_rows}
    selected_hashes = {r['_image_id']: r['_source_image_sha256'] for r in selected_rows}
    if len(selected_rows) != len(selected_ids):
        raise ValueError('Locked main selection contains duplicate source IDs')
    if {r['image_id'] for r in records} != selected_ids:
        raise ValueError('Main metadata does not cover the locked source selection exactly')
    if any(r['source_image_sha256'] != selected_hashes[r['image_id']] for r in records):
        raise ValueError('Generator image_id-to-source hash disagrees with locked selection')
    record_keys = [(r['image_id'], r['condition'], r['level']) for r in records]
    if len(set(record_keys)) != len(records):
        raise ValueError('Duplicate main image metadata records')
    if any(r['config_sha256'] != frozen_hash or r['source_manifest_sha256'] != selection_hash for r in records):
        raise ValueError('Main records use different frozen config/selection; do not mix runs')

    clean_files, degraded_files = image_index(paths['clean_root']), image_index(paths['degraded_root'] / 'main')
    counts = Counter(main_metadata_instances=len(records), main_unique_sources=len(selected_ids),
                     textvqa_unique_sources=len(clean_ids), textocr_sources=len(textocr['imgs']))
    candidates, excluded = defaultdict(list), []
    for i, r in enumerate(sorted(records, key=lambda x: (x['image_id'], x['condition'], x['level']))):
        image_id = str(r['image_id'])
        if not re.fullmatch(r'[A-Za-z0-9_-]+', image_id):
            raise ValueError('Unsupported source ID characters; do not infer or normalize IDs')
        def exclude(reason, detail=''):
            counts['excluded_' + reason] += 1
            excluded.append(dict(image_id=image_id, condition=r['condition'], level=r['level'],
                                 reason=reason, detail=str(detail)))
        if image_id not in clean_ids:
            exclude('not_in_textvqa'); continue
        if image_id not in textocr['imgs'] or not textocr['img2Anns'].get(image_id):
            exclude('not_in_textocr_or_no_gt'); continue
        counts['intersection_before_file_geometry_checks'] += 1
        try:
            clean = resolve_recorded_path(r['source_image_path'], paths['clean_root'], clean_files,
                                          r['source_image_sha256'], 'clean')
            degraded = resolve_recorded_path(r['output_path'], paths['degraded_root'], degraded_files,
                                             r['output_sha256'], 'degraded')
            size, degraded_size = image_size(clean), image_size(degraded)
            meta = textocr['imgs'][image_id]
            if size != (int(meta['width']), int(meta['height'])):
                raise ValueError('Clean pixels and TextOCR GT dimensions differ')
            h, source_visible, ops = source_geometry(r, size, degraded_size)
        except (ValueError, OSError, KeyError, np.linalg.LinAlgError) as exc:
            exclude('file_or_geometry_invalid', exc); continue
        candidates[image_id].append(dict(record=r, clean=clean, degraded=degraded, size=size,
                                         matrix=h, source_visible=source_visible, ops=ops))
        if (i + 1) % 250 == 0:
            print('AUDIT:', i + 1, '/', len(records), flush=True)

    rows = []
    for image_id, variants in sorted(candidates.items()):
        roi = box(0, 0, *variants[0]['size'])
        for v in variants:
            roi = roi.intersection(v['source_visible'])
        gt_clean, removed_ids = gt_for_source(image_id, textocr, roi, config, counts)
        if not gt_clean['valid']:
            counts['excluded_source_no_usable_valid_gt'] += 1
            for v in variants:
                excluded.append(dict(image_id=image_id, condition=v['record']['condition'],
                                     level=v['record']['level'], reason='no_usable_valid_gt', detail=''))
            continue
        for v in variants:
            r, h = v['record'], v['matrix']
            gt_degraded = {key: [dict(item, geometry=mapping(warp_geometry(shape(item['geometry']), h)))
                                for item in gt_clean[key]] for key in ('valid', 'ignored')}
            variant_id = hashlib.sha256(canonical_bytes([image_id, r['condition'], r['level'],
                                                        r['output_sha256'], r['config_sha256']])).hexdigest()[:24]
            rows.append(dict(schema_version=SCHEMA_VERSION, source_image_id=image_id, variant_id=variant_id,
                             clean_relpath=v['clean'].relative_to(paths['clean_root']).as_posix(),
                             degraded_relpath=v['degraded'].relative_to(paths['degraded_root']).as_posix(),
                             clean_sha256=r['source_image_sha256'], degraded_sha256=r['output_sha256'],
                             clean_size=list(v['size']), degraded_size=list(v['size']),
                             condition=r['condition'], severity=r['level'], operations=v['ops'],
                             has_perspective='perspective' in v['ops'], homography=h.tolist(),
                             roi_clean=mapping(roi), roi_degraded=mapping(warp_geometry(roi, h)),
                             gt_clean=gt_clean, gt_degraded=gt_degraded, removed_gt_ids=removed_ids,
                             source_config_sha256=r['config_sha256'],
                             severity_basis=r.get('severity_basis', 'not_recorded')))
    rows.sort(key=lambda r: (r['source_image_id'], r['variant_id']))
    if not rows:
        raise RuntimeError('Empty valid intersection. STOP and inspect audit; no full inference.')
    counts['evaluated_instances'] = len(rows)
    counts['evaluated_unique_sources'] = len({r['source_image_id'] for r in rows})
    counts['perspective_instances'] = sum(r['has_perspective'] for r in rows)
    protocol = dict(schema_version=SCHEMA_VERSION, matching='maximum_cardinality_polygon_iou',
                    match_iou=config['match_iou'], word_coverage=config['coverage_threshold'],
                    min_prediction_roi_fraction=config['min_prediction_roi_fraction'],
                    ignore_prediction_overlap=config['ignore_prediction_overlap'],
                    ignore_transcripts=config['ignore_transcripts'], min_word_visibility=config['min_word_visibility'],
                    score_policy='No additional score filtering/sweep; preserve each detector fixed postprocessing',
                    geometry_policy='common source field of view across variants; same valid annotation IDs in clean/degraded',
                    split_policy='baseline_all_sources_no_training; no unseen-data claim; split before later fine-tuning')
    manifest = output / 'shared_manifest_v1.jsonl'
    digest = write_locked(manifest, b''.join(canonical_bytes(row) for row in rows))
    protocol_hash = write_locked(output / 'evaluation_protocol.json', canonical_bytes(protocol))
    write_locked(output / 'shared_manifest_v1.sha256', (digest + '\n').encode())
    counts['excluded_instances_total'] = len(excluded)
    audit = dict(counts=dict(counts), manifest_sha256=digest, protocol_sha256=protocol_hash,
                 generator_frozen_sha256=frozen_hash, generator_selection_sha256=selection_hash,
                 actual_generator_policy=policy, textocr_sha256=file_sha256(paths['textocr_json']),
                 textocr_schema=textocr_schema,
                 clean_annotation_sha256=file_sha256(paths['clean_json']),
                 gt_identity_policy=protocol['geometry_policy'], scope='MAIN only; excludes calibration and ablation')
    write_locked(output / 'data_audit.json', canonical_bytes(audit))
    write_locked(output / 'excluded.jsonl', b''.join(canonical_bytes(row) for row in excluded))
    clean_by_source = {r['source_image_id']: r for r in rows}
    index_tables = {
        'clean_index.csv': [dict(image_id=s, relative_path=r['clean_relpath'], sha256=r['clean_sha256'])
                            for s, r in sorted(clean_by_source.items())],
        'degraded_index.csv': [dict(image_id=r['source_image_id'], variant_id=r['variant_id'],
                                   relative_path=r['degraded_relpath'], sha256=r['degraded_sha256'],
                                   geometry='homography' if r['has_perspective'] else 'identity',
                                   homography_json=json.dumps(r['homography'], separators=(',', ':')))
                               for r in rows],
    }
    for name, records_out in index_tables.items():
        buf = io.StringIO(newline='')
        writer = csv.DictWriter(buf, fieldnames=list(records_out[0]), lineterminator='\n')
        writer.writeheader(); writer.writerows(records_out)
        write_locked(output / name, buf.getvalue().encode('utf-8'))
    locations = dict(clean_root=str(paths['clean_root']), degraded_root=str(paths['degraded_root']))
    # Machine-local roots are outside the portable frozen manifest.
    (output / 'dataset_locations.json').write_bytes(canonical_bytes(locations))
    save_gt_overlays(rows, locations, output / 'gt_overlays', config['overlay_sources'])
    report = '\n'.join(['# Phase 2 data audit', '', f'Manifest SHA256: `{digest}`', '',
                        'MAIN only; IDs from generator metadata, verified source/output hashes.', '',
                        *[f'- {k}: {v}' for k, v in sorted(counts.items())], '',
                        'Cyan: valid field of view. Green: valid GT. Gray: ignored/cropped GT.',
                        'Review overlays before full inference. This run does not fine-tune or generate images.'])
    (output / 'data_audit.md').write_text(report + '\n', encoding='utf-8')
    print(json.dumps(audit, ensure_ascii=False, indent=2), flush=True)
    return dict(manifest=str(manifest), manifest_sha256=digest, output_root=str(output),
                locations=str(output / 'dataset_locations.json'), protocol=str(output / 'evaluation_protocol.json'),
                counts=dict(counts))


# ===== PUBLIC CELL INVOCATION: copy this entire cell unchanged to the Paddle notebook =====
# Only actual root/path overrides may differ per machine; freeze/copy the produced manifest bytes.
SHARED_CONFIG = dict(DEFAULT_SHARED_CONFIG)
# Leave these blank on Kaggle when there is exactly one matching real file.
# SHARED_CONFIG['clean_json'] = '...actual file from discovery...'
# SHARED_CONFIG['textocr_json'] = '...actual file from discovery...'
# SHARED_CONFIG['main_image_manifest'] = '...actual main/image_manifest.csv...'
SHARED_RESULT = build_shared_manifest(SHARED_CONFIG)
print('SHARED_RESULT:', json.dumps(SHARED_RESULT, ensure_ascii=False, indent=2))


## Audit và snapshot dữ liệu trước inference
Xem `data_audit.json`, `excluded.jsonl`, `gt_overlays/`; cyan là field of view hợp lệ,
xanh là GT được chấm, xám là ignored. Đặc biệt kiểm tra các ảnh perspective.
Hash dưới đây là dấu nhận dạng dữ liệu/protocol, không phải nhãn reproduce thành công.
Luồng mặc định tiếp tục tự động; overlays được giữ lại để kiểm tra cùng kết quả cuối.


In [ ]:
from IPython.display import display, Image as DisplayImage
SHARED=Path(SHARED_RESULT['output_root'])
print('Manifest SHA256:',SHARED_RESULT['manifest_sha256'])
print('Protocol SHA256:',file_sha256(SHARED/'evaluation_protocol.json'))
print((SHARED/'data_audit.md').read_text())
for p in sorted((SHARED/'gt_overlays').glob('*.jpg'))[:4]:
    print(p.name);display(DisplayImage(filename=str(p),width=800))
data_identity=dict(manifest_sha256=SHARED_RESULT['manifest_sha256'],
                   protocol_sha256=file_sha256(SHARED/'evaluation_protocol.json'),
                   validation_mode='automatic_audit_then_smoke')
if EXPECTED_SHARED_MANIFEST_SHA256:
    assert EXPECTED_SHARED_MANIFEST_SHA256==data_identity['manifest_sha256'],'Manifest differs from optional pinned hash.'
if EXPECTED_SHARED_PROTOCOL_SHA256:
    assert EXPECTED_SHARED_PROTOCOL_SHA256==data_identity['protocol_sha256'],'Protocol differs from optional pinned hash.'
write_locked(ART/'data_identity.json',canonical_bytes(data_identity))
print('Data snapshot locked automatically:',data_identity)


## Setup stack Monkey trong môi trường riêng
Phần cài đặt tái sử dụng workflow Phase 1 đã chạy thành công của Tuấn, pin đúng commits/HF revisions.
Không tải ICDAR2015, không chạy training, không thay kernel PyTorch/Paddle. Venv từ output cũ
gắn read-only ở `/kaggle/input` không mặc định relocatable; notebook tạo env ở đường dẫn working hiện tại.
Nếu dùng lại cùng session, installer/checkpoint có thể tái sử dụng theo lock.


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str) + '\n', encoding='utf-8')

def run(cmd, log_name, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    log = ART / 'logs' / log_name
    print('RUN:', subprocess.list2cmdline(cmd), flush=True)
    with log.open('w', encoding='utf-8') as f:
        f.write(json.dumps(dict(cmd=cmd, cwd=str(cwd)), ensure_ascii=False) + '\n')
        proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1)
        for line in proc.stdout:
            f.write(line)
            f.flush()
            print(line, end='', flush=True)
        rc = proc.wait()
    if rc:
        save_json(ART / 'blocker.json', dict(stage=log_name, exit_code=rc, log=str(log),
                                           time=datetime.now(timezone.utc).isoformat()))
        raise RuntimeError(f'BLOCKED: exit {rc}; xem {log}. Không tiếp tục evaluation.')
    return log

def output(cmd, cwd=None, env=None):
    return subprocess.check_output([str(x) for x in cmd], cwd=cwd, env=env, text=True).strip()

def script(name, source):
    p = ART / 'scripts' / name
    compile(source, str(p), 'exec')
    p.write_text(source, encoding='utf-8')
    return p

def block(message):
    save_json(ART / 'blocker.json', dict(message=message, time=datetime.now(timezone.utc).isoformat()))
    raise RuntimeError(message)

assert platform.system() == 'Linux', 'Notebook dành cho Kaggle/Linux GPU; không Run All trên Windows.'
assert shutil.which('git') and shutil.which('nvidia-smi'), 'Bật GPU T4 và Internet trong Kaggle Settings.'
run(['nvidia-smi'], '00_nvidia_smi.log')
run([sys.executable, '-c', 'import sys,platform; print(sys.version); print(platform.platform())'], '00_host.log')

# Tìm nvcc thực, không suy ra toolkit từ CUDA Version trong nvidia-smi.
nvcc_options = []
if CUDA_HOME_OVERRIDE:
    nvcc_options = [Path(CUDA_HOME_OVERRIDE) / 'bin/nvcc']
else:
    nvcc_options = sorted(Path('/usr/local').glob('cuda*/bin/nvcc'))
    if shutil.which('nvcc'):
        nvcc_options.append(Path(shutil.which('nvcc')))
toolkits = {}
for candidate in nvcc_options:
    if candidate.is_file():
        text = output([candidate, '--version'])
        match = re.search(r'release (\d+\.\d+)', text)
        if match:
            toolkits[str(candidate.resolve())] = (match.group(1), candidate.resolve().parent.parent)
print('CUDA toolkits:', toolkits)
preferred = [v for v in toolkits.values() if v[0] == '12.8']
if not preferred and ALLOW_CUDA_MINOR_MISMATCH:
    preferred = sorted([v for v in toolkits.values() if v[0].startswith('12.')], reverse=True)
if not preferred:
    block('Không có nvcc 12.8. Dừng trước khi cài stack lớn. Kiểm tra CUDA_HOME_OVERRIDE; '
          'nếu chỉ có toolkit 12.x khác, có thể bật ALLOW_CUDA_MINOR_MISMATCH để thử môi trường adapted '
          '(có rủi ro build/runtime). H200 cũng phải kiểm tra toolkit, không tự sửa lỗi phiên bản.')
NVCC_VERSION, CUDA_HOME_PATH = preferred[0]
save_json(ART / 'preflight.json', dict(toolkits=toolkits, selected_cuda_home=str(CUDA_HOME_PATH),
                                     nvcc_version=NVCC_VERSION, adapted_cuda=NVCC_VERSION != '12.8'))
print('Selected toolkit:', CUDA_HOME_PATH, NVCC_VERSION)


In [ ]:
assert platform.system()=='Linux'
if GPU_IDS=='auto':
    visible=output(['nvidia-smi','--query-gpu=index','--format=csv,noheader']).splitlines()
    GPU_IDS=','.join(v.strip() for v in visible[:2])
assert GPU_IDS and all(v.strip().isdigit() for v in GPU_IDS.split(','))
print('Selected GPUs:',GPU_IDS)
bootstrap = ROOT / 'uv_bootstrap'
if not (bootstrap / 'uv').exists():
    run([sys.executable, '-m', 'pip', 'install', '--target', bootstrap, 'uv'], '01_bootstrap_uv.log')
uv_env = os.environ.copy()
uv_env['PYTHONPATH'] = str(bootstrap)
uv_cmd = [sys.executable, '-m', 'uv']
if not PY.is_file():
    run(uv_cmd + ['python', 'install', '3.11'], '01_python311.log', env=uv_env)
    run(uv_cmd + ['venv', '--python', '3.11', '--seed', VENV], '01_venv.log', env=uv_env)
assert output([PY, '-c', 'import sys; print(f"{sys.version_info.major}.{sys.version_info.minor}")']) == '3.11'

ENV = os.environ.copy()
ENV.pop('PYTHONPATH', None)
ENV.update(PATH=str(VENV / 'bin') + os.pathsep + ENV['PATH'], VIRTUAL_ENV=str(VENV),
           PYTHONPATH=str(MMOCR_ROOT),
           CUDA_HOME=str(CUDA_HOME_PATH), MAX_JOBS=str(MAX_BUILD_JOBS),
           CUDA_VISIBLE_DEVICES=GPU_IDS, MPLBACKEND='Agg', TOKENIZERS_PARALLELISM='false',
           PIP_DISABLE_PIP_VERSION_CHECK='1', PIP_NO_INPUT='1', PIP_NO_CACHE_DIR='1')
# Chỉ checkpoint/code chính thức đã khóa revision/hash; scope trong các subprocess của env này.
# MMEngine cũ gọi torch.load mà không chỉ định weights_only; tương thích với torch >=2.6.
ENV['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

constraints = ART / 'constraints.txt'
constraints.write_text('\n'.join([
    'torch==2.9.0', 'torchvision==0.24.0', 'numpy==1.26.4', 'setuptools<81',
    'mmengine==0.10.7', 'mmcv==2.0.1', 'mmdet==3.1.0', 'mmocr==1.0.1',
    'transformers==4.57.1', 'opencv-python==4.10.0.84',
]) + '\n')
ENV['PIP_CONSTRAINT'] = str(constraints)

lock_path = ART / 'source_lock.json'
locks = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if REPO.exists() and not (REPO / '.git').is_dir():
    block(f'{REPO} tồn tại nhưng không phải Git checkout; không ghi đè.')
if not REPO.exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', '--filter=blob:none', MONKEY_URL, REPO], '02_clone_monkey.log')
target_ref = locks.get('monkey_commit', MONKEY_REF)
run(['git', 'checkout', '--detach', target_ref], '02_checkout_monkey.log', cwd=REPO)
commit = output(['git', 'rev-parse', 'HEAD'], cwd=REPO)
locks['monkey_commit'] = commit
save_json(lock_path, locks)
(ART / 'git_commit.txt').write_text(commit + '\n')
assert (ADDON / 'install.sh').is_file(), 'Repo không chứa detection/mmocr/install.sh; kiểm tra revision.'
shutil.copy2(ADDON / 'install.sh', ART / 'source/install.sh')
print((ADDON / 'install.sh').read_text())


In [ ]:
run([PY, '-m', 'pip', 'install', 'torch==2.9.0', 'torchvision==0.24.0',
     '--index-url', 'https://download.pytorch.org/whl/cu128'], '03_torch.log', env=ENV)
run([PY, '-m', 'pip', 'install', 'numpy==1.26.4', 'setuptools<81', 'wheel', 'ninja',
     'mmengine==0.10.7', 'opencv-python==4.10.0.84'], '03_build_dependencies.log', env=ENV)
gpu_probe = script('gpu_probe.py', """
import json, torch
assert torch.cuda.is_available(), 'Torch không sử dụng được GPU; kiểm tra driver/runtime.'
x = torch.ones((8, 8), device='cuda')
assert (x @ x).sum().item() == 512
gpus = [dict(name=torch.cuda.get_device_name(i), capability=torch.cuda.get_device_capability(i))
        for i in range(torch.cuda.device_count())]
print(json.dumps(dict(torch=torch.__version__, cuda=torch.version.cuda, gpus=gpus)))
print('ARCH_LIST=' + ';'.join(sorted({'.'.join(map(str, torch.cuda.get_device_capability(i)))
                                     for i in range(torch.cuda.device_count())})))
""")
probe_log = run([PY, gpu_probe], '03_gpu_probe.log', env=ENV)
arch_match = re.search(r'ARCH_LIST=(.+)', probe_log.read_text())
assert arch_match
ENV['TORCH_CUDA_ARCH_LIST'] = arch_match.group(1).strip()
print('Build CUDA architectures:', ENV['TORCH_CUDA_ARCH_LIST'])

MMCV_SRC = ROOT / 'mmcv_src'
if not MMCV_SRC.exists():
    run(['git', 'clone', '--branch', 'v2.0.1', '--depth', '1',
         'https://github.com/open-mmlab/mmcv.git', MMCV_SRC], '04_clone_mmcv.log', env=ENV)
assert output(['git', 'describe', '--tags', '--exact-match'], cwd=MMCV_SRC) == 'v2.0.1'
setup = MMCV_SRC / 'setup.py'
original_setup = output(['git', 'show', 'HEAD:setup.py'], cwd=MMCV_SRC)
setup.write_text(original_setup.replace('-std=c++14', '-std=c++17'))
run(['git', 'diff'], '04_mmcv_cxx17.patch', cwd=MMCV_SRC)
build_env = dict(ENV, MMCV_WITH_OPS='1', FORCE_CUDA='1')
run([PY, '-m', 'pip', 'install', '-v', '-e', MMCV_SRC, '--no-build-isolation'],
    '04_build_mmcv.log', env=build_env)
locks['mmcv_commit'] = output(['git', 'rev-parse', 'HEAD'], cwd=MMCV_SRC)
save_json(lock_path, locks)


In [ ]:
# Official installer tạo nested ./mmocr và copy configs/patches. Chạy một lần mỗi source commit.
installed_marker = ADDON / '.phase1_installed_commit'
if not installed_marker.exists():
    if MMOCR_ROOT.exists():
        block('Nested MMOCR đã tồn tại nhưng chưa có marker. Xem logs cài trước; không tự overwrite. '
              'Dùng Kaggle session sạch để cài lại nếu installer trước bị gián đoạn.')
    run(['bash', 'install.sh'], '05_official_install.log', cwd=ADDON, env=ENV)
    assert (MMOCR_ROOT / 'tools/test.py').is_file()
    installed_marker.write_text(commit)
else:
    assert installed_marker.read_text().strip() == commit, 'Source khác marker; cần session/source root mới.'
assert (MMOCR_ROOT / '.git').is_dir()
locks['mmocr_commit'] = output(['git', 'rev-parse', 'HEAD'], cwd=MMOCR_ROOT)
save_json(lock_path, locks)

run([PY, '-m', 'pip', 'install', 'mmdet==3.1.0', 'transformers==4.57.1',
     'huggingface_hub', 'safetensors', 'einops', 'timm', 'Pillow', 'scipy',
     'shapely', 'pyclipper', 'imgaug', 'ftfy', 'lmdb'], '05_detection_dependencies.log', env=ENV)
run([PY, '-m', 'pip', 'install', '-e', MMOCR_ROOT], '05_mmocr_editable.log', env=ENV)
run(['git', 'diff', '--stat'], '05_mmocr_patch_stat.log', cwd=MMOCR_ROOT)
run(['git', 'diff'], '05_mmocr.patch', cwd=MMOCR_ROOT)

ops_probe = script('ops_probe.py', """
import importlib.metadata as im, json
import torch, mmcv, mmengine, mmdet, mmocr, transformers
from mmcv.ops import nms, ModulatedDeformConv2d
expected = {'torch':'2.9.0', 'torchvision':'0.24.0', 'mmcv':'2.0.1', 'mmengine':'0.10.7',
            'mmdet':'3.1.0', 'mmocr':'1.0.1', 'transformers':'4.57.1', 'numpy':'1.26.4'}
actual = {k: im.version(k) for k in expected}
for k, v in expected.items():
    assert actual[k].split('+')[0] == v, (k, actual[k], v)
boxes = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]], device='cuda')
dets, keep = nms(boxes, torch.tensor([.9,.8], device='cuda'), .5)
assert keep.numel() == 1
op = ModulatedDeformConv2d(2, 2, 3, padding=1).cuda()
y = op(torch.randn(1,2,8,8,device='cuda'), torch.zeros(1,18,8,8,device='cuda'),
       torch.ones(1,9,8,8,device='cuda'))
assert y.shape == (1,2,8,8) and torch.isfinite(y).all()
torch.cuda.synchronize()
print(json.dumps(actual, indent=2)); print('CUDA nms + modulated deform conv: OK')
""")
run([PY, ops_probe], '06_imports_cuda_ops.log', cwd=MMOCR_ROOT, env=ENV)
run([PY, '-m', 'pip', 'freeze'], 'env_versions.txt', env=ENV)
shutil.copy2(ART / 'logs/env_versions.txt', ART / 'env_versions.txt')


In [ ]:
download_script = script('download_artifacts.py', """
import hashlib, json, sys
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
root, lock_path, det_repo, det_file, backbone_repo = sys.argv[1:]
root, lock_path = Path(root), Path(lock_path)
locks = json.loads(lock_path.read_text())
api = HfApi()
items = [(det_repo, 'detector_hf_revision'), (backbone_repo, 'backbone_hf_revision')]
for repo, key in items:
    if key not in locks:
        locks[key] = api.model_info(repo).sha
        lock_path.write_text(json.dumps(locks, indent=2))
files = api.list_repo_files(det_repo, revision=locks['detector_hf_revision'])
assert det_file in files, f'Không có {det_file}. Các pth có thật: {[x for x in files if x.endswith(".pth")]}'
det = hf_hub_download(det_repo, det_file, revision=locks['detector_hf_revision'],
                      local_dir=root/'model_weight')
snapshot_download(backbone_repo, revision=locks['backbone_hf_revision'],
                  local_dir=root/'pretrained/monkeyocrv2_as')
assert (root/'pretrained/monkeyocrv2_as/config.json').is_file()
print('Detector:', det)
""")
run([PY, download_script, MMOCR_ROOT, lock_path, DETECTOR_REPO, DETECTOR_FILENAME, BACKBONE_REPO],
    '07_download_artifacts.log', env=ENV)
locks = json.loads(lock_path.read_text())
CHECKPOINT = MMOCR_ROOT / 'model_weight' / DETECTOR_FILENAME
CONFIG = MMOCR_ROOT / CONFIG_REL
assert CHECKPOINT.is_file() and CONFIG.is_file()
shutil.copy2(CONFIG, ART / 'source' / CONFIG.name)
hashes = dict(detector=dict(path=str(CHECKPOINT), sha256=sha256_file(CHECKPOINT),
                           bytes=CHECKPOINT.stat().st_size,
                           url=f'https://huggingface.co/{DETECTOR_REPO}/resolve/{locks["detector_hf_revision"]}/{DETECTOR_FILENAME}'),
              config=dict(path=str(CONFIG), sha256=sha256_file(CONFIG)), backbone={})
for p in sorted((MMOCR_ROOT / 'pretrained/monkeyocrv2_as').rglob('*')):
    if p.is_file() and '.cache' not in p.parts:
        hashes['backbone'][str(p.relative_to(MMOCR_ROOT))] = dict(sha256=sha256_file(p), bytes=p.stat().st_size)
save_json(ART / 'artifact_hashes.json', hashes)
(ART / 'checkpoint_sha256.txt').write_text(hashes['detector']['sha256'] + '  ' + DETECTOR_FILENAME + '\n')
print(json.dumps(hashes['detector'], indent=2))

assert file_sha256(CHECKPOINT)==EXPECTED_CHECKPOINT_SHA256,'Checkpoint bytes differ from verified Phase 1.'
assert file_sha256(CONFIG)==EXPECTED_CONFIG_SHA256,'Official model config differs from verified Phase 1.'
assert json.loads((ART/'source_lock.json').read_text())==EXPECTED_SOURCE_LOCK
print('Verified: same checkpoint/config/source revisions as Phase 1.')


## Inference và evaluator chung
Scripts dưới đây được nhúng trong notebook, đồng thời có source `.py` để đóng gói cho Paddle.
Evaluator không dùng `HmeanIOUMetric` sweep của Phase 1. Chấm polygon word-level theo protocol v1
chung và diagnostic word coverage; giữ nguyên đầu ra detector, không chọn threshold theo test.


In [ ]:
(ART/"scripts"/'shared_manifest.py').write_text('"""Public, detector-independent intersection cell. No model, training or degradation generation.\n\nSupported input metadata schema: the four v2_10k generator notebooks supplied by Tuan.\nAll paths are discovered from real files; source IDs come from JSON/CSV, never degraded filenames.\nThe frozen manifest is root-independent. Copy the resulting bytes to Paddle; do not rebuild it.\n"""\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport re\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\nfrom shapely.geometry import Polygon, box, mapping, shape\ntry:\n    from shapely import transform as array_geometry_transform\nexcept ImportError:\n    array_geometry_transform = None\nfrom shapely.ops import transform as legacy_geometry_transform\n\nSCHEMA_VERSION = \'textdet-paired-v1\'\nGENERATOR_PROTOCOL = \'TEXTVQA_REALISTIC_DEGRADATION_V2_10K\'\nKNOWN_OPERATIONS = {\'motion_camera_shake\', \'defocus_blur\', \'lowlight_sensor_noise\',\n                    \'overexposure_glare\', \'low_resolution\', \'perspective\', \'jpeg_compression\'}\nDEFAULT_SHARED_CONFIG = dict(\n    input_root=\'/kaggle/input\', output_root=\'/kaggle/working/textdet_shared_v1\',\n    clean_json=\'\', textocr_json=\'\', main_image_manifest=\'\',\n    clean_images_root=\'\', degraded_root=\'\',\n    ignore_transcripts=[\'.\', \'###\', \'\'], min_word_visibility=0.80,\n    min_prediction_roi_fraction=0.50, ignore_prediction_overlap=0.50,\n    match_iou=0.50, coverage_threshold=0.50, overlay_sources=12,\n)\n\n\ndef canonical_bytes(obj):\n    return (json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\'),\n                       allow_nan=False) + \'\\n\').encode(\'utf-8\')\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for data in iter(lambda: f.read(1024 * 1024), b\'\'):\n            digest.update(data)\n    return digest.hexdigest()\n\n\ndef write_locked(path, data):\n    path = Path(path)\n    if path.exists() and path.read_bytes() != data:\n        raise RuntimeError(f\'Frozen artifact changed: {path}. Use a new output revision; do not overwrite.\')\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_bytes(data)\n    return hashlib.sha256(data).hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'))\n\n\ndef describe_textocr_schema(raw):\n    """Small diagnostics, not the entire large annotation JSON or transcripts."""\n    report = dict(root_type=type(raw).__name__)\n    if isinstance(raw, dict):\n        report[\'top_level_keys\'] = sorted(raw.keys())\n        fields = {}\n        for key, value in raw.items():\n            field = dict(type=type(value).__name__)\n            if isinstance(value, (dict, list)):\n                field[\'length\'] = len(value)\n                example = next(iter(value.values()), None) if isinstance(value, dict) else next(iter(value), None)\n                if isinstance(example, dict):\n                    field[\'record_keys\'] = sorted(example.keys())\n                elif example is not None:\n                    field[\'element_type\'] = type(example).__name__\n            fields[key] = field\n        report[\'fields\'] = fields\n    return report\n\n\ndef normalize_textocr(raw):\n    """Canonicalize explicit IDs and rebuild the redundant image->annotations index.\n\n    The source of the mapping is each annotation\'s image_id, NOT filenames, list positions,\n    or a guessed association. Polygon points remain unchanged. Existing usable indexes\n    are checked against these foreign keys before their order is retained.\n    """\n    report = describe_textocr_schema(raw)\n    if not isinstance(raw, dict):\n        raise ValueError(f\'TextOCR must be an object: {report}\')\n    def choose(primary, alias):\n        if primary in raw:\n            return primary, raw[primary]\n        if alias in raw:\n            return alias, raw[alias]\n        raise ValueError(f\'Missing {primary}/{alias}; actual keys={report.get("top_level_keys")}\')\n    image_key, image_records = choose(\'imgs\', \'images\')\n    annotation_key, annotation_records = choose(\'anns\', \'annotations\')\n    def canonical_id(value, label):\n        if isinstance(value, bool) or not isinstance(value, (str, int)) or value == \'\':\n            raise ValueError(f\'{label}: missing/non-scalar explicit ID: {value!r}\')\n        return str(value)  # Do not strip leading zeroes, change case, or parse filename stems.\n    def index_records(records, label):\n        if not isinstance(records, (dict, list)):\n            raise ValueError(f\'{label} must be dict/list; actual type={type(records).__name__}\')\n        result = {}\n        iterator = records.items() if isinstance(records, dict) else ((None, r) for r in records)\n        for key, record in iterator:\n            if not isinstance(record, dict):\n                raise ValueError(f\'{label}: expected object records, got {type(record).__name__}\')\n            item_id = canonical_id(record.get(\'id\') if key is None else key, label)\n            if \'id\' in record and canonical_id(record[\'id\'], label) != item_id:\n                raise ValueError(f\'{label}: dictionary key disagrees with record.id for {item_id}\')\n            if item_id in result:\n                raise ValueError(f\'{label}: duplicate explicit ID {item_id}\')\n            result[item_id] = record\n        return result\n    images = index_records(image_records, image_key)\n    annotations = index_records(annotation_records, annotation_key)\n    if not images or not annotations:\n        raise ValueError(\'TextOCR images/annotations are empty; test metadata without GT is not usable here\')\n    rebuilt = defaultdict(list)\n    for ann_id, ann in annotations.items():\n        image_id = canonical_id(ann.get(\'image_id\'), f\'annotation {ann_id}.image_id\')\n        if image_id not in images:\n            raise ValueError(f\'Annotation {ann_id} references unknown image_id {image_id}\')\n        if \'points\' not in ann:\n            raise ValueError(f\'Annotation {ann_id} has no polygon points; actual record keys={sorted(ann)}. \'\n                             \'Do not substitute bbox or a recognition/VQA annotation as polygon GT.\')\n        rebuilt[image_id].append(ann_id)\n    provided_keys = [key for key in (\'img2Anns\', \'imgToAnns\') if key in raw]\n    selected_index, validated_keys = None, []\n    unusable_index_types = {}\n    for key in provided_keys:\n        supplied = raw[key]\n        if not isinstance(supplied, dict):\n            # This is only a redundant index; explicit annotation foreign keys remain authoritative.\n            unusable_index_types[key] = type(supplied).__name__\n            continue\n        normalized = {}\n        for image_id, ids in supplied.items():\n            image_id = canonical_id(image_id, key)\n            if image_id not in images or not isinstance(ids, list):\n                raise ValueError(f\'{key}: unknown image ID or non-list annotation IDs for {image_id}\')\n            ann_ids = [canonical_id(item, key) for item in ids]\n            if len(set(ann_ids)) != len(ann_ids):\n                raise ValueError(f\'{key}: duplicate annotation references for {image_id}\')\n            if set(ann_ids) != set(rebuilt.get(image_id, [])):\n                raise ValueError(f\'{key}: links disagree with annotations.image_id for {image_id}; do not guess mapping\')\n            normalized[image_id] = ann_ids\n        if any(set(normalized.get(image_id, [])) != set(ids) for image_id, ids in rebuilt.items()):\n            raise ValueError(f\'{key}: index omits annotations present in anns; inconsistent GT needs audit\')\n        if selected_index is None:\n            selected_index = normalized\n        validated_keys.append(key)\n    if selected_index is None:\n        selected_index = {image_id: sorted(ids) for image_id, ids in rebuilt.items()}\n    report.update(image_field=image_key, annotation_field=annotation_key,\n                  canonical_images=len(images), canonical_annotations=len(annotations),\n                  validated_index_keys=validated_keys, non_dictionary_index_types=unusable_index_types,\n                  mapping_source=(\'validated \' + validated_keys[0]) if validated_keys else \'rebuilt from anns[*].image_id\',\n                  polygon_source=\'unchanged points; no bbox/filename fallback\')\n    return dict(imgs=images, anns=annotations, img2Anns=selected_index), report\n\n\ndef csv_rows(path):\n    with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n        return list(csv.DictReader(f))\n\n\ndef one_file(explicit, candidates, label):\n    paths = [Path(explicit)] if explicit else sorted(set(Path(p) for p in candidates))\n    if len(paths) != 1 or not paths[0].is_file():\n        raise RuntimeError(f\'{label}: expected one real file; candidates={[str(p) for p in paths]}. \'\n                           \'Set the corresponding SHARED_CONFIG path explicitly. Do not guess.\')\n    return paths[0].resolve()\n\n\ndef discover(config):\n    root = Path(config[\'input_root\']).resolve()\n    if not root.is_dir():\n        raise FileNotFoundError(f\'Attach the three Kaggle datasets; missing {root}\')\n    # Names are defined by the supplied generator and the original annotation releases.\n    clean = one_file(config[\'clean_json\'], root.rglob(\'TextVQA_0.5.1_train.json\'), \'clean_json\')\n    textocr = one_file(config[\'textocr_json\'], root.rglob(\'TextOCR_0.1_train.json\'), \'textocr_json\')\n    main_candidates = [p for p in root.rglob(\'image_manifest.csv\') if p.parent.name == \'main\']\n    main = one_file(config[\'main_image_manifest\'], main_candidates, \'main_image_manifest\')\n    return dict(clean_json=clean, textocr_json=textocr, main_manifest=main,\n                clean_root=Path(config[\'clean_images_root\']).resolve() if config[\'clean_images_root\'] else clean.parent,\n                degraded_root=Path(config[\'degraded_root\']).resolve() if config[\'degraded_root\'] else main.parent.parent)\n\n\ndef image_index(root):\n    result = defaultdict(list)\n    for p in sorted(Path(root).rglob(\'*\')):\n        if p.is_file() and p.suffix.lower() in {\'.jpg\', \'.jpeg\', \'.png\', \'.webp\', \'.bmp\'}:\n            result[p.name].append(p.resolve())\n    return result\n\n\ndef resolve_recorded_path(recorded, root, index, expected_hash, kind):\n    """Rebind a recorded path by exact suffix/basename and mandatory content hash.\n\n    This does NOT infer image IDs from filenames. ID -> recorded file is in generator metadata.\n    """\n    if not recorded or not expected_hash or len(expected_hash) != 64:\n        raise ValueError(f\'{kind}: missing recorded path or SHA256\')\n    root = Path(root).resolve()\n    parts = PurePosixPath(str(recorded).replace(\'\\\\\', \'/\')).parts\n    candidates = []\n    direct = Path(recorded)\n    if direct.is_file() and direct.resolve().is_relative_to(root):\n        candidates.append(direct.resolve())\n    if kind == \'degraded\':\n        anchors = [i for i in range(len(parts) - 1) if parts[i:i + 2] == (\'main\', \'images\')]\n        if len(anchors) == 1:\n            rebound = root.joinpath(*parts[anchors[0]:])\n            if rebound.is_file():\n                candidates.append(rebound.resolve())\n    # A unique exact basename is allowed only with the metadata hash check below.\n    if not candidates:\n        candidates = index.get(parts[-1], []) if parts else []\n    candidates = sorted(set(candidates))\n    if len(candidates) != 1:\n        raise ValueError(f\'{kind}: ambiguous/missing file for metadata path {recorded}: {candidates}\')\n    p = candidates[0]\n    if not p.is_relative_to(root) or file_sha256(p) != expected_hash:\n        raise ValueError(f\'{kind}: bytes/path do not match generator metadata: {p}\')\n    return p\n\n\ndef image_size(path):\n    with Image.open(path) as im:\n        if im.getexif().get(274, 1) not in (None, 1):\n            raise ValueError(\'Non-identity EXIF orientation; coordinate frame needs explicit audit\')\n        size = im.size\n        im.convert(\'RGB\').load()  # Decode every paired image; detect damaged files before inference.\n    return size\n\n\ndef homography_from_corners(width, height, corners):\n    src = np.array([[0., 0.], [width - 1., 0.], [width - 1., height - 1.], [0., height - 1.]])\n    dst = np.asarray(corners, dtype=float)\n    if dst.shape != (4, 2) or not np.isfinite(dst).all():\n        raise ValueError(\'Perspective corners must be finite 4x2, in generator source-corner order\')\n    a, b = [], []\n    for (x, y), (u, v) in zip(src, dst):\n        a.extend([[x, y, 1, 0, 0, 0, -u*x, -u*y], [0, 0, 0, x, y, 1, -v*x, -v*y]])\n        b.extend([u, v])\n    h = np.append(np.linalg.solve(np.array(a), np.array(b)), 1).reshape(3, 3)\n    if not np.isfinite(h).all() or np.linalg.cond(h) > 1e12:\n        raise ValueError(\'Unstable homography\')\n    projected = project_points(src, h)\n    if np.max(np.abs(projected - dst)) > 1e-4:\n        raise ValueError(\'Corner reprojection check failed\')\n    return h\n\n\ndef project_points(points, h):\n    points = np.asarray(points, dtype=float).reshape(-1, 2)\n    q = np.column_stack([points, np.ones(len(points))]) @ np.asarray(h).T\n    if not np.isfinite(q).all() or np.any(np.abs(q[:, 2]) < 1e-10):\n        raise ValueError(\'Homography crosses projective horizon\')\n    return q[:, :2] / q[:, 2, None]\n\n\ndef warp_geometry(geom, h):\n    h = np.asarray(h, dtype=float)\n    def project(x, y, z=None):\n        xa, ya = np.asarray(x), np.asarray(y)\n        den = h[2, 0]*xa + h[2, 1]*ya + h[2, 2]\n        if np.any(np.abs(den) < 1e-10):\n            raise ValueError(\'Projective horizon intersects polygon\')\n        return ((h[0, 0]*xa + h[0, 1]*ya + h[0, 2])/den,\n                (h[1, 0]*xa + h[1, 1]*ya + h[1, 2])/den)\n    out = (array_geometry_transform(geom, lambda coordinates: project_points(coordinates, h))\n           if array_geometry_transform is not None else legacy_geometry_transform(project, geom))\n    if not out.is_valid:\n        out = out.buffer(0)\n    if out.is_empty or out.area <= 0:\n        raise ValueError(\'Empty warped geometry\')\n    return out\n\n\ndef source_geometry(record, size, output_size):\n    if size != output_size:\n        raise ValueError(\'Generator expects unchanged canvas. Unknown resize/crop; metadata audit needed.\')\n    params = record[\'params\']\n    if isinstance(params, str):\n        params = json.loads(params)\n    if not isinstance(params, list) or not params:\n        raise ValueError(\'Missing per-image operation metadata\')\n    ops = [p[\'operation\'] for p in params]\n    if not set(ops) <= KNOWN_OPERATIONS:\n        raise ValueError(f\'Unsupported geometric/unknown operations: {set(ops)-KNOWN_OPERATIONS}\')\n    perspectives = [p for p in params if p[\'operation\'] == \'perspective\']\n    if len(perspectives) > 1:\n        raise ValueError(\'Multiple perspective transforms require explicit composition\')\n    w, h = size\n    matrix = homography_from_corners(w, h, perspectives[0][\'corners\']) if perspectives else np.eye(3)\n    canvas = box(0, 0, w, h)\n    # Reflected border pixels are outside the source field of view, and must not create GT/FPs.\n    source_visible = canvas.intersection(warp_geometry(canvas, np.linalg.inv(matrix)))\n    if source_visible.is_empty or source_visible.area / canvas.area < 0.50:\n        raise ValueError(\'Implausible visible field of view\')\n    return matrix, source_visible, ops\n\n\ndef annotation_geometry(annotation):\n    arr = np.asarray(annotation[\'points\'], dtype=float)\n    if arr.size < 6 or arr.size % 2 or not np.isfinite(arr).all():\n        raise ValueError(\'Invalid TextOCR polygon coordinates\')\n    geom = Polygon(arr.reshape(-1, 2))\n    repaired = not geom.is_valid\n    if repaired:\n        geom = geom.buffer(0)\n    if geom.is_empty or geom.area <= 0 or geom.geom_type not in {\'Polygon\', \'MultiPolygon\'}:\n        raise ValueError(\'Unusable TextOCR polygon\')\n    return geom, repaired\n\n\ndef gt_for_source(image_id, textocr, roi, config, audit_counts):\n    valid, ignored, removed_ids = [], [], []\n    for ann_id in textocr[\'img2Anns\'].get(image_id, []):\n        ann = textocr[\'anns\'][str(ann_id)]\n        if str(ann.get(\'image_id\', image_id)) != image_id:\n            raise ValueError(\'TextOCR img2Anns / anns image_id mismatch\')\n        try:\n            geom, repaired = annotation_geometry(ann)\n        except (ValueError, KeyError):\n            audit_counts[\'unusable_gt_annotations\'] += 1\n            removed_ids.append(str(ann_id))\n            continue\n        audit_counts[\'repaired_gt_annotations\'] += int(repaired)\n        clipped = geom.intersection(roi)\n        visibility = clipped.area / geom.area\n        if clipped.is_empty or clipped.area <= 0:\n            audit_counts[\'gt_outside_common_field_of_view\'] += 1\n            removed_ids.append(str(ann_id))\n            continue\n        unreadable = str(ann.get(\'utf8_string\', \'\')).strip() in config[\'ignore_transcripts\']\n        explicit_ignore = ann.get(\'ignore\', False) is True or ann.get(\'ignored\', False) is True\n        partial = visibility < config[\'min_word_visibility\']\n        entry = dict(ann_id=str(ann_id), geometry=mapping(clipped),\n                     original_height=float(geom.bounds[3] - geom.bounds[1]), visibility=float(visibility))\n        if unreadable or explicit_ignore or partial:\n            entry[\'ignore_reason\'] = \'unreadable\' if unreadable or explicit_ignore else \'cropped_below_visibility\'\n            ignored.append(entry)\n            audit_counts[\'gt_ignored_\' + entry[\'ignore_reason\']] += 1\n        else:\n            valid.append(entry)\n    return dict(valid=valid, ignored=ignored), removed_ids\n\n\ndef overlay(geometry, drawing, color, width=2):\n    if geometry.geom_type == \'Polygon\':\n        drawing.line(list(geometry.exterior.coords), fill=color, width=width)\n    elif hasattr(geometry, \'geoms\'):\n        for part in geometry.geoms:\n            overlay(part, drawing, color, width)\n\n\ndef save_gt_overlays(rows, locations, output, limit):\n    output.mkdir(parents=True, exist_ok=True)\n    chosen = sorted(rows, key=lambda r: (not r[\'has_perspective\'], r[\'source_image_id\'], r[\'variant_id\']))[:limit]\n    for i, row in enumerate(chosen):\n        for cond in (\'clean\', \'degraded\'):\n            p = Path(locations[cond + \'_root\']) / row[cond + \'_relpath\']\n            with Image.open(p) as im:\n                im = im.convert(\'RGB\')\n                draw = ImageDraw.Draw(im)\n                overlay(shape(row[\'roi_\' + cond]), draw, \'#00BFFF\', 3)\n                for item in row[\'gt_\' + cond][\'ignored\']:\n                    overlay(shape(item[\'geometry\']), draw, \'#AAAAAA\')\n                for item in row[\'gt_\' + cond][\'valid\']:\n                    overlay(shape(item[\'geometry\']), draw, \'#00FF00\')\n                im.thumbnail((1280, 1280))\n                im.save(output / f\'{i:02d}_{row["source_image_id"]}_{cond}.jpg\', quality=88)\n\n\ndef build_shared_manifest(config=None):\n    config = dict(DEFAULT_SHARED_CONFIG, **(config or {}))\n    output = Path(config[\'output_root\']).resolve()\n    output.mkdir(parents=True, exist_ok=True)\n    assert 0 < config[\'min_word_visibility\'] <= 1\n    paths = discover(config)\n    print(\'DISCOVERED (real files):\', json.dumps({k: str(v) for k, v in paths.items()}, indent=2), flush=True)\n    records = csv_rows(paths[\'main_manifest\'])\n    required = {\'image_id\', \'condition\', \'level\', \'params\', \'output_path\', \'source_image_path\',\n                \'source_image_sha256\', \'output_sha256\', \'config_sha256\', \'source_manifest_sha256\'}\n    if not records or not required <= records[0].keys():\n        raise ValueError(f\'Unsupported generator CSV schema; required={sorted(required)}\')\n    clean_json = read_json(paths[\'clean_json\'])\n    if not isinstance(clean_json.get(\'data\'), list):\n        raise ValueError(\'Unsupported TextVQA schema: expected data list\')\n    clean_ids = {str(r[\'image_id\']) for r in clean_json[\'data\']}\n    del clean_json\n    textocr_raw = read_json(paths[\'textocr_json\'])\n    schema_path = output / \'textocr_schema.json\'\n    schema_path.write_bytes(canonical_bytes(describe_textocr_schema(textocr_raw)))\n    print(\'TEXTOCR RAW SCHEMA:\', json.dumps(describe_textocr_schema(textocr_raw), indent=2), flush=True)\n    try:\n        textocr, textocr_schema = normalize_textocr(textocr_raw)\n    except ValueError as exc:\n        raise ValueError(f\'{exc}. Schema diagnostics: {schema_path}\') from exc\n    schema_path.write_bytes(canonical_bytes(textocr_schema))\n    del textocr_raw\n    print(\'TEXTOCR MAPPING:\', textocr_schema[\'mapping_source\'], flush=True)\n    print(\'SCHEMAS:\', dict(main_columns=sorted(records[0]), main_records=len(records),\n                          textvqa_unique_ids=len(clean_ids), textocr_images=len(textocr[\'imgs\'])), flush=True)\n\n    # Verify the actual frozen config against every selected generator record.\n    frozen_path = paths[\'degraded_root\'] / \'config/frozen_degradation_config.json\'\n    policy_path = paths[\'degraded_root\'] / \'metadata/split_policy.json\'\n    eval_path = paths[\'degraded_root\'] / \'metadata/locked_eval_manifest.csv\'\n    if not all(p.is_file() for p in (frozen_path, policy_path, eval_path)):\n        raise ValueError(\'Missing frozen config / split policy / locked_eval_manifest. Inspect input; do not guess.\')\n    frozen, policy = read_json(frozen_path), read_json(policy_path)\n    if frozen.get(\'protocol\') != GENERATOR_PROTOCOL or policy.get(\'protocol\') != GENERATOR_PROTOCOL:\n        raise ValueError(\'Not the supplied v2_10k generator protocol; needs an explicit schema adapter\')\n    frozen_hash, selection_hash = file_sha256(frozen_path), file_sha256(eval_path)\n    if frozen.get(\'eval_manifest_sha256\') != selection_hash:\n        raise ValueError(\'Frozen config / selection manifest hash mismatch\')\n    if frozen.get(\'train_annotation_sha256\') != file_sha256(paths[\'clean_json\']):\n        raise ValueError(\'Attached clean TextVQA JSON is not the source used by the generator\')\n    selected_rows = csv_rows(eval_path)\n    if not selected_rows or \'_source_image_sha256\' not in selected_rows[0]:\n        raise ValueError(\'Locked selection lacks source-image hashes; cannot verify ID-to-clean mapping\')\n    selected_ids = {r[\'_image_id\'] for r in selected_rows}\n    selected_hashes = {r[\'_image_id\']: r[\'_source_image_sha256\'] for r in selected_rows}\n    if len(selected_rows) != len(selected_ids):\n        raise ValueError(\'Locked main selection contains duplicate source IDs\')\n    if {r[\'image_id\'] for r in records} != selected_ids:\n        raise ValueError(\'Main metadata does not cover the locked source selection exactly\')\n    if any(r[\'source_image_sha256\'] != selected_hashes[r[\'image_id\']] for r in records):\n        raise ValueError(\'Generator image_id-to-source hash disagrees with locked selection\')\n    record_keys = [(r[\'image_id\'], r[\'condition\'], r[\'level\']) for r in records]\n    if len(set(record_keys)) != len(records):\n        raise ValueError(\'Duplicate main image metadata records\')\n    if any(r[\'config_sha256\'] != frozen_hash or r[\'source_manifest_sha256\'] != selection_hash for r in records):\n        raise ValueError(\'Main records use different frozen config/selection; do not mix runs\')\n\n    clean_files, degraded_files = image_index(paths[\'clean_root\']), image_index(paths[\'degraded_root\'] / \'main\')\n    counts = Counter(main_metadata_instances=len(records), main_unique_sources=len(selected_ids),\n                     textvqa_unique_sources=len(clean_ids), textocr_sources=len(textocr[\'imgs\']))\n    candidates, excluded = defaultdict(list), []\n    for i, r in enumerate(sorted(records, key=lambda x: (x[\'image_id\'], x[\'condition\'], x[\'level\']))):\n        image_id = str(r[\'image_id\'])\n        if not re.fullmatch(r\'[A-Za-z0-9_-]+\', image_id):\n            raise ValueError(\'Unsupported source ID characters; do not infer or normalize IDs\')\n        def exclude(reason, detail=\'\'):\n            counts[\'excluded_\' + reason] += 1\n            excluded.append(dict(image_id=image_id, condition=r[\'condition\'], level=r[\'level\'],\n                                 reason=reason, detail=str(detail)))\n        if image_id not in clean_ids:\n            exclude(\'not_in_textvqa\'); continue\n        if image_id not in textocr[\'imgs\'] or not textocr[\'img2Anns\'].get(image_id):\n            exclude(\'not_in_textocr_or_no_gt\'); continue\n        counts[\'intersection_before_file_geometry_checks\'] += 1\n        try:\n            clean = resolve_recorded_path(r[\'source_image_path\'], paths[\'clean_root\'], clean_files,\n                                          r[\'source_image_sha256\'], \'clean\')\n            degraded = resolve_recorded_path(r[\'output_path\'], paths[\'degraded_root\'], degraded_files,\n                                             r[\'output_sha256\'], \'degraded\')\n            size, degraded_size = image_size(clean), image_size(degraded)\n            meta = textocr[\'imgs\'][image_id]\n            if size != (int(meta[\'width\']), int(meta[\'height\'])):\n                raise ValueError(\'Clean pixels and TextOCR GT dimensions differ\')\n            h, source_visible, ops = source_geometry(r, size, degraded_size)\n        except (ValueError, OSError, KeyError, np.linalg.LinAlgError) as exc:\n            exclude(\'file_or_geometry_invalid\', exc); continue\n        candidates[image_id].append(dict(record=r, clean=clean, degraded=degraded, size=size,\n                                         matrix=h, source_visible=source_visible, ops=ops))\n        if (i + 1) % 250 == 0:\n            print(\'AUDIT:\', i + 1, \'/\', len(records), flush=True)\n\n    rows = []\n    for image_id, variants in sorted(candidates.items()):\n        roi = box(0, 0, *variants[0][\'size\'])\n        for v in variants:\n            roi = roi.intersection(v[\'source_visible\'])\n        gt_clean, removed_ids = gt_for_source(image_id, textocr, roi, config, counts)\n        if not gt_clean[\'valid\']:\n            counts[\'excluded_source_no_usable_valid_gt\'] += 1\n            for v in variants:\n                excluded.append(dict(image_id=image_id, condition=v[\'record\'][\'condition\'],\n                                     level=v[\'record\'][\'level\'], reason=\'no_usable_valid_gt\', detail=\'\'))\n            continue\n        for v in variants:\n            r, h = v[\'record\'], v[\'matrix\']\n            gt_degraded = {key: [dict(item, geometry=mapping(warp_geometry(shape(item[\'geometry\']), h)))\n                                for item in gt_clean[key]] for key in (\'valid\', \'ignored\')}\n            variant_id = hashlib.sha256(canonical_bytes([image_id, r[\'condition\'], r[\'level\'],\n                                                        r[\'output_sha256\'], r[\'config_sha256\']])).hexdigest()[:24]\n            rows.append(dict(schema_version=SCHEMA_VERSION, source_image_id=image_id, variant_id=variant_id,\n                             clean_relpath=v[\'clean\'].relative_to(paths[\'clean_root\']).as_posix(),\n                             degraded_relpath=v[\'degraded\'].relative_to(paths[\'degraded_root\']).as_posix(),\n                             clean_sha256=r[\'source_image_sha256\'], degraded_sha256=r[\'output_sha256\'],\n                             clean_size=list(v[\'size\']), degraded_size=list(v[\'size\']),\n                             condition=r[\'condition\'], severity=r[\'level\'], operations=v[\'ops\'],\n                             has_perspective=\'perspective\' in v[\'ops\'], homography=h.tolist(),\n                             roi_clean=mapping(roi), roi_degraded=mapping(warp_geometry(roi, h)),\n                             gt_clean=gt_clean, gt_degraded=gt_degraded, removed_gt_ids=removed_ids,\n                             source_config_sha256=r[\'config_sha256\'],\n                             severity_basis=r.get(\'severity_basis\', \'not_recorded\')))\n    rows.sort(key=lambda r: (r[\'source_image_id\'], r[\'variant_id\']))\n    if not rows:\n        raise RuntimeError(\'Empty valid intersection. STOP and inspect audit; no full inference.\')\n    counts[\'evaluated_instances\'] = len(rows)\n    counts[\'evaluated_unique_sources\'] = len({r[\'source_image_id\'] for r in rows})\n    counts[\'perspective_instances\'] = sum(r[\'has_perspective\'] for r in rows)\n    protocol = dict(schema_version=SCHEMA_VERSION, matching=\'maximum_cardinality_polygon_iou\',\n                    match_iou=config[\'match_iou\'], word_coverage=config[\'coverage_threshold\'],\n                    min_prediction_roi_fraction=config[\'min_prediction_roi_fraction\'],\n                    ignore_prediction_overlap=config[\'ignore_prediction_overlap\'],\n                    ignore_transcripts=config[\'ignore_transcripts\'], min_word_visibility=config[\'min_word_visibility\'],\n                    score_policy=\'No additional score filtering/sweep; preserve each detector fixed postprocessing\',\n                    geometry_policy=\'common source field of view across variants; same valid annotation IDs in clean/degraded\',\n                    split_policy=\'baseline_all_sources_no_training; no unseen-data claim; split before later fine-tuning\')\n    manifest = output / \'shared_manifest_v1.jsonl\'\n    digest = write_locked(manifest, b\'\'.join(canonical_bytes(row) for row in rows))\n    protocol_hash = write_locked(output / \'evaluation_protocol.json\', canonical_bytes(protocol))\n    write_locked(output / \'shared_manifest_v1.sha256\', (digest + \'\\n\').encode())\n    counts[\'excluded_instances_total\'] = len(excluded)\n    audit = dict(counts=dict(counts), manifest_sha256=digest, protocol_sha256=protocol_hash,\n                 generator_frozen_sha256=frozen_hash, generator_selection_sha256=selection_hash,\n                 actual_generator_policy=policy, textocr_sha256=file_sha256(paths[\'textocr_json\']),\n                 textocr_schema=textocr_schema,\n                 clean_annotation_sha256=file_sha256(paths[\'clean_json\']),\n                 gt_identity_policy=protocol[\'geometry_policy\'], scope=\'MAIN only; excludes calibration and ablation\')\n    write_locked(output / \'data_audit.json\', canonical_bytes(audit))\n    write_locked(output / \'excluded.jsonl\', b\'\'.join(canonical_bytes(row) for row in excluded))\n    clean_by_source = {r[\'source_image_id\']: r for r in rows}\n    index_tables = {\n        \'clean_index.csv\': [dict(image_id=s, relative_path=r[\'clean_relpath\'], sha256=r[\'clean_sha256\'])\n                            for s, r in sorted(clean_by_source.items())],\n        \'degraded_index.csv\': [dict(image_id=r[\'source_image_id\'], variant_id=r[\'variant_id\'],\n                                   relative_path=r[\'degraded_relpath\'], sha256=r[\'degraded_sha256\'],\n                                   geometry=\'homography\' if r[\'has_perspective\'] else \'identity\',\n                                   homography_json=json.dumps(r[\'homography\'], separators=(\',\', \':\')))\n                               for r in rows],\n    }\n    for name, records_out in index_tables.items():\n        buf = io.StringIO(newline=\'\')\n        writer = csv.DictWriter(buf, fieldnames=list(records_out[0]), lineterminator=\'\\n\')\n        writer.writeheader(); writer.writerows(records_out)\n        write_locked(output / name, buf.getvalue().encode(\'utf-8\'))\n    locations = dict(clean_root=str(paths[\'clean_root\']), degraded_root=str(paths[\'degraded_root\']))\n    # Machine-local roots are outside the portable frozen manifest.\n    (output / \'dataset_locations.json\').write_bytes(canonical_bytes(locations))\n    save_gt_overlays(rows, locations, output / \'gt_overlays\', config[\'overlay_sources\'])\n    report = \'\\n\'.join([\'# Phase 2 data audit\', \'\', f\'Manifest SHA256: `{digest}`\', \'\',\n                        \'MAIN only; IDs from generator metadata, verified source/output hashes.\', \'\',\n                        *[f\'- {k}: {v}\' for k, v in sorted(counts.items())], \'\',\n                        \'Cyan: valid field of view. Green: valid GT. Gray: ignored/cropped GT.\',\n                        \'Review overlays before full inference. This run does not fine-tune or generate images.\'])\n    (output / \'data_audit.md\').write_text(report + \'\\n\', encoding=\'utf-8\')\n    print(json.dumps(audit, ensure_ascii=False, indent=2), flush=True)\n    return dict(manifest=str(manifest), manifest_sha256=digest, output_root=str(output),\n                locations=str(output / \'dataset_locations.json\'), protocol=str(output / \'evaluation_protocol.json\'),\n                counts=dict(counts))\n',encoding="utf-8")
(ART/"scripts"/'evaluate_shared.py').write_text('"""One external TextOCR polygon evaluator for BOTH detectors; no score sweep/test tuning."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nfrom collections import defaultdict, deque\nfrom pathlib import Path\n\nimport numpy as np\nfrom shapely.geometry import Polygon, shape\nfrom shapely.ops import unary_union\n\n\ndef read_jsonl(path):\n    with Path(path).open(encoding=\'utf-8\') as f:\n        return [json.loads(line) for line in f if line.strip()]\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024*1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef maximum_matching(adjacency, num_predictions):\n    """Maximum cardinality bipartite matching, not order-dependent greedy matching."""\n    prediction_to_gt = [-1] * num_predictions\n    gt_to_prediction = [-1] * len(adjacency)\n    for start in range(len(adjacency)):\n        queue, seen_gt, previous_prediction = deque([start]), {start}, {}\n        free = -1\n        while queue and free < 0:\n            gt_index = queue.popleft()\n            for pred_index in adjacency[gt_index]:\n                if pred_index in previous_prediction:\n                    continue\n                previous_prediction[pred_index] = gt_index\n                owner = prediction_to_gt[pred_index]\n                if owner < 0:\n                    free = pred_index\n                    break\n                if owner not in seen_gt:\n                    seen_gt.add(owner); queue.append(owner)\n        while free >= 0:\n            gt_index = previous_prediction[free]\n            old_prediction = gt_to_prediction[gt_index]\n            prediction_to_gt[free] = gt_index\n            gt_to_prediction[gt_index] = free\n            free = old_prediction\n    return [(g, p) for p, g in enumerate(prediction_to_gt) if g >= 0]\n\n\ndef rates(counts):\n    tp, gt, pred, covered = [float(x) for x in counts]\n    precision = tp / pred if pred else 0.\n    recall = tp / gt if gt else 0.\n    return dict(precision=precision, recall=recall,\n                hmean=2*precision*recall/(precision+recall) if precision+recall else 0.,\n                word_coverage_recall=covered/gt if gt else 0.)\n\n\ndef evaluate_image(gt_entries, roi_mapping, prediction, protocol):\n    valid_gt = gt_entries[\'valid\']\n    ignored_gt = gt_entries[\'ignored\']\n    gt_shapes = [shape(x[\'geometry\']) for x in valid_gt]\n    roi = shape(roi_mapping)\n    ignore_union = unary_union([shape(x[\'geometry\']) for x in ignored_gt])\n    polygons, scores = prediction[\'polygons\'], prediction[\'scores\']\n    if len(polygons) != len(scores):\n        raise ValueError(\'Mismatched polygon/score counts\')\n    kept, invalid_count, ignored_roi, ignored_annotations = [], 0, 0, 0\n    for original_index, (coords, score) in enumerate(zip(polygons, scores)):\n        if not math.isfinite(float(score)) or not 0 <= score <= 1:\n            raise ValueError(\'Non-finite/out-of-range model confidence\')\n        array = np.asarray(coords, dtype=float)\n        if array.size < 6 or array.size % 2 or not np.isfinite(array).all():\n            invalid_count += 1\n            continue  # Invalid output is counted as an FP below, not silently discarded.\n        geom = Polygon(array.reshape(-1, 2))\n        if not geom.is_valid:\n            geom = geom.buffer(0)\n        if geom.is_empty or geom.area <= 0:\n            invalid_count += 1\n            continue\n        clipped = geom.intersection(roi)\n        if clipped.is_empty or clipped.area / geom.area < protocol[\'min_prediction_roi_fraction\']:\n            ignored_roi += 1\n            continue\n        if not ignore_union.is_empty and clipped.intersection(ignore_union).area / clipped.area > protocol[\'ignore_prediction_overlap\']:\n            ignored_annotations += 1\n            continue\n        kept.append((original_index, clipped))\n    adjacency = []\n    for g in gt_shapes:\n        overlaps = []\n        for j, (_, p) in enumerate(kept):\n            inter = g.intersection(p).area\n            iou = inter / (g.area + p.area - inter) if inter else 0.\n            if iou >= protocol[\'match_iou\']:\n                overlaps.append((j, iou))\n        adjacency.append([j for j, _ in sorted(overlaps, key=lambda x: (-x[1], x[0]))])\n    pairs = maximum_matching(adjacency, len(kept))\n    matched_gt = {g for g, _ in pairs}\n    matched_predictions = {p for _, p in pairs}\n    union = unary_union([g for _, g in kept])\n    covered = {i for i, g in enumerate(gt_shapes)\n               if not union.is_empty and union.intersection(g).area / g.area >= protocol[\'word_coverage\']}\n    counts = [len(pairs), len(gt_shapes), len(kept) + invalid_count, len(covered)]\n    size_bins = {}\n    for label, lo, hi in [(\'small_lt16px\', 0, 16), (\'medium_16_32px\', 16, 32), (\'large_ge32px\', 32, float(\'inf\'))]:\n        indices = {i for i, g in enumerate(valid_gt) if lo <= g[\'original_height\'] < hi}\n        size_bins[label] = dict(gt=len(indices), matched=len(indices & matched_gt), covered=len(indices & covered))\n    return dict(counts=counts, **rates(counts),\n                missed_ann_ids=[valid_gt[i][\'ann_id\'] for i in range(len(gt_shapes)) if i not in matched_gt],\n                matched_ann_ids=[valid_gt[i][\'ann_id\'] for i in sorted(matched_gt)],\n                fp_prediction_indices=[kept[i][0] for i in range(len(kept)) if i not in matched_predictions],\n                invalid_prediction_count=invalid_count, ignored_outside_roi=ignored_roi,\n                ignored_on_annotations=ignored_annotations,\n                predicted_roi_area_fraction=float(union.area / roi.area) if not union.is_empty else 0.,\n                size_bins=size_bins)\n\n\ndef bootstrap_paired(source_vectors, repetitions=1000, seed=42):\n    matrix = np.asarray(source_vectors, dtype=float)\n    if len(matrix) < 2 or repetitions <= 0:\n        return dict(method=\'cluster bootstrap by source_image_id\', available=False)\n    rng = np.random.default_rng(seed)\n    samples = defaultdict(list)\n    for _ in range(repetitions):\n        aggregate = matrix[rng.integers(0, len(matrix), len(matrix))].sum(axis=0)\n        clean, degraded = rates(aggregate[:4]), rates(aggregate[4:])\n        for key in clean:\n            samples[key].append(100*(degraded[key]-clean[key]))\n    return dict(method=\'cluster bootstrap by source_image_id\', available=True,\n                repetitions=repetitions, seed=seed,\n                delta_pp_95ci={key: [float(x) for x in np.quantile(values, [.025, .975])]\n                               for key, values in samples.items()})\n\n\ndef evaluate(manifest, predictions, protocol_path, output_dir, bootstrap=1000):\n    output = Path(output_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    rows, preds = read_jsonl(manifest), read_jsonl(predictions)\n    protocol = json.loads(Path(protocol_path).read_text())\n    manifest_hash = sha256(manifest)\n    pred_map = {p[\'sample_id\']: p for p in preds}\n    if len(pred_map) != len(preds):\n        raise ValueError(\'Duplicate prediction sample_id\')\n    clean_by_id = {}\n    for row in rows:\n        source = row[\'source_image_id\']\n        if source in clean_by_id and row[\'gt_clean\'] != clean_by_id[source][\'gt_clean\']:\n            raise ValueError(\'Clean GT identity differs across variants; common-source GT required\')\n        clean_by_id[source] = row\n    expected = {f\'clean::{s}\' for s in clean_by_id} | {f\'deg::{r["variant_id"]}\' for r in rows}\n    if expected != set(pred_map):\n        raise ValueError(f\'Incomplete prediction set: missing={len(expected-set(pred_map))}, extra={len(set(pred_map)-expected)}\')\n    if any(p.get(\'manifest_sha256\') != manifest_hash for p in preds):\n        raise ValueError(\'Predictions belong to a different frozen manifest\')\n    models = {p[\'model\'] for p in preds}\n    if len(models) != 1:\n        raise ValueError(\'Do not mix model predictions\')\n    clean_results, degraded_results = {}, []\n    size_totals = {c: defaultdict(lambda: np.zeros(3)) for c in (\'clean\', \'degraded\')}\n    for source, row in sorted(clean_by_id.items()):\n        key = f\'clean::{source}\'\n        result = evaluate_image(row[\'gt_clean\'], row[\'roi_clean\'], pred_map[key], protocol)\n        clean_results[source] = dict(sample_id=key, source_image_id=source, condition=\'clean\', **result)\n        for bucket, vals in result[\'size_bins\'].items():\n            size_totals[\'clean\'][bucket] += [vals[\'gt\'], vals[\'matched\'], vals[\'covered\']]\n    for i, row in enumerate(rows):\n        key = f\'deg::{row["variant_id"]}\'\n        result = evaluate_image(row[\'gt_degraded\'], row[\'roi_degraded\'], pred_map[key], protocol)\n        degraded_results.append(dict(sample_id=key, source_image_id=row[\'source_image_id\'],\n                                     variant_id=row[\'variant_id\'], condition=\'degraded\',\n                                     family=row[\'condition\'], severity=row[\'severity\'],\n                                     has_perspective=row[\'has_perspective\'], **result))\n        for bucket, vals in result[\'size_bins\'].items():\n            size_totals[\'degraded\'][bucket] += [vals[\'gt\'], vals[\'matched\'], vals[\'covered\']]\n        if (i+1) % 250 == 0:\n            print(\'EVALUATED:\', i+1, \'/\', len(rows), flush=True)\n    def summarize(items):\n        counts = np.asarray([r[\'counts\'] for r in items], dtype=float).sum(axis=0)\n        return dict(instances=len(items), counts=dict(zip((\'tp\',\'gt\',\'pred\',\'covered\'), counts.tolist())), **rates(counts))\n    clean_summary = summarize(list(clean_results.values()))\n    degraded_summary = summarize(degraded_results)\n    by_source = defaultdict(list)\n    for r in degraded_results:\n        by_source[r[\'source_image_id\']].append(r)\n    vectors, pairs = [], []\n    for source, variants in sorted(by_source.items()):\n        c = clean_results[source]\n        d_counts = np.mean([r[\'counts\'] for r in variants], axis=0)\n        vectors.append(list(c[\'counts\']) + d_counts.tolist())\n        d = rates(d_counts)\n        pair = dict(source_image_id=source, variants=len(variants),\n                    clean_counts=c[\'counts\'], degraded_mean_counts=d_counts.tolist())\n        for metric in (\'precision\', \'recall\', \'hmean\', \'word_coverage_recall\'):\n            pair[metric + \'_clean\'] = c[metric]\n            pair[metric + \'_degraded\'] = d[metric]\n            pair[metric + \'_delta_pp\'] = 100*(d[metric]-c[metric])\n        pairs.append(pair)\n    matrix = np.asarray(vectors)\n    paired_clean, paired_degraded = rates(matrix[:, :4].sum(axis=0)), rates(matrix[:, 4:].sum(axis=0))\n    stratified = {}\n    for group in sorted({(r[\'family\'], r[\'severity\'], r[\'has_perspective\']) for r in degraded_results}):\n        subset = [r for r in degraded_results if (r[\'family\'], r[\'severity\'], r[\'has_perspective\']) == group]\n        paired_baseline = [clean_results[r[\'source_image_id\']] for r in subset]\n        c, d = summarize(paired_baseline), summarize(subset)\n        stratified[str(group)] = dict(clean_paired=c, degraded=d,\n                                      delta_pp={k:100*(d[k]-c[k]) for k in paired_clean})\n    latency = {}\n    for condition in (\'clean\', \'degraded\'):\n        values = [float(p[\'inference_ms_with_decode_and_postprocess\']) for p in preds\n                  if p.get(\'condition\') == condition and \'inference_ms_with_decode_and_postprocess\' in p]\n        latency[condition] = (dict(available=True, samples=len(values), mean_ms=float(np.mean(values)),\n                                   p50_ms=float(np.quantile(values, .5)), p95_ms=float(np.quantile(values, .95)))\n                              if values else dict(available=False))\n    result = dict(evaluation_type=\'external_TextVQA_TextOCR_word_level_polygon_evaluation\',\n                  model=next(iter(models)), manifest_sha256=manifest_hash,\n                  evaluator_sha256=sha256(__file__),\n                  protocol_sha256=sha256(protocol_path), unique_sources=len(clean_by_id),\n                  degraded_variants=len(rows), score_policy=protocol[\'score_policy\'],\n                  latency_with_decode_and_postprocess=latency,\n                  clean_unique=clean_summary, degraded_instance=degraded_summary,\n                  paired_source_weighted_micro=dict(clean=paired_clean, degraded=paired_degraded,\n                      delta_pp={k:100*(paired_degraded[k]-paired_clean[k]) for k in paired_clean}),\n                  macro_source_mean={k:dict(clean=float(np.mean([p[k+\'_clean\'] for p in pairs])),\n                                           degraded=float(np.mean([p[k+\'_degraded\'] for p in pairs])))\n                                     for k in paired_clean},\n                  bootstrap=bootstrap_paired(vectors, bootstrap), strata=stratified,\n                  size_bins={cond:{key:dict(gt=int(v[0]),matched=int(v[1]),covered=int(v[2]),\n                                           recall=float(v[1]/v[0]) if v[0] else None)\n                                   for key,v in buckets.items()} for cond,buckets in size_totals.items()},\n                  limitations=[\'Synthetic degradation, not captured degraded images\',\n                               \'TextOCR train is included in MonkeyOCRv2 pretraining sources; no unseen-data claim\',\n                               \'Word GT versus possibly line predictions; coverage is diagnostic and can be gamed by large polygons\',\n                               \'Full-set baseline audit; do not later fine-tune and claim this entire set is an untouched test\'])\n    (output/\'metrics.json\').write_text(json.dumps(result,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\')\n    with (output/\'per_image.jsonl\').open(\'w\',encoding=\'utf-8\') as f:\n        for row in list(clean_results.values())+degraded_results:\n            f.write(json.dumps(row,ensure_ascii=False)+\'\\n\')\n    with (output/\'paired_source_metrics.csv\').open(\'w\',encoding=\'utf-8\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(pairs[0]));writer.writeheader();writer.writerows(pairs)\n    worst=sorted(pairs,key=lambda r:r[\'recall_delta_pp\'])[:20]\n    (output/\'worst_recall_sources.json\').write_text(json.dumps(worst,indent=2)+\'\\n\')\n    table=[\'# Clean versus degraded — checkpoint baseline\',\'\',\n           \'External TextVQA images + TextOCR word GT; not paper-official Hmean. No fine-tuning.\', \'\',\n           \'| Metric | Clean paired (%) | Degraded paired (%) | Delta (pp) | 95% cluster-bootstrap CI (pp) |\',\n           \'|---|---:|---:|---:|---|\']\n    for key in paired_clean:\n        interval=result[\'bootstrap\'].get(\'delta_pp_95ci\',{}).get(key)\n        table.append(f\'| {key} | {100*paired_clean[key]:.3f} | {100*paired_degraded[key]:.3f} | \'\n                     f\'{100*(paired_degraded[key]-paired_clean[key]):+.3f} | {interval} |\')\n    table += [\'\', f\'Sources: {len(clean_by_id)}; degraded variants: {len(rows)}.\',\n              f\'Manifest SHA256: `{manifest_hash}`.\', \'\',\n              \'Delta = degraded - clean. CI excluding zero indicates a directional difference under this protocol;\',\n              \'effect size in percentage points determines practical magnitude. No preset conclusion of a large drop.\',\n              \'Review word-vs-line granularity, giant boxes, FP/FN and geometric masks alongside these scores.\']\n    (output/\'clean_vs_degraded.md\').write_text(\'\\n\'.join(table)+\'\\n\',encoding=\'utf-8\')\n    return result\n\n\nif __name__ == \'__main__\':\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--manifest\',required=True);p.add_argument(\'--predictions\',required=True)\n    p.add_argument(\'--protocol\',required=True);p.add_argument(\'--output-dir\',required=True)\n    p.add_argument(\'--bootstrap\',type=int,default=1000)\n    a=p.parse_args()\n    results=evaluate(a.manifest,a.predictions,a.protocol,a.output_dir,a.bootstrap)\n    print(json.dumps(results[\'paired_source_weighted_micro\'],indent=2))\n',encoding="utf-8")
(ART/"scripts"/'run_monkey.py').write_text('"""Checkpoint-only, resumable sharded inference: one independent process per visible GPU.\n\nGT and homographies are never passed to the detector. Model outputs are in input-image pixels.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef read_jsonl(path):\n    with Path(path).open(encoding=\'utf-8\') as f:\n        return [json.loads(s) for s in f if s.strip()]\n\n\ndef json_line(obj):\n    return json.dumps(obj,sort_keys=True,ensure_ascii=False,allow_nan=False)+\'\\n\'\n\n\ndef build_tasks(manifest,locations,smoke_sources=0):\n    rows=read_jsonl(manifest)\n    if smoke_sources:\n        # Deterministic sample spanning perspective and identity; IDs remain metadata-derived.\n        source_groups={}\n        for r in rows:\n            source_groups.setdefault(r[\'source_image_id\'],[]).append(r)\n        geom=sorted(s for s,rs in source_groups.items() if any(r[\'has_perspective\'] for r in rs))\n        plain=sorted(set(source_groups)-set(geom))\n        chosen=[]\n        while len(chosen)<smoke_sources and (geom or plain):\n            if geom:chosen.append(geom.pop(0))\n            if plain and len(chosen)<smoke_sources:chosen.append(plain.pop(0))\n        rows=[r for r in rows if r[\'source_image_id\'] in set(chosen)]\n    unique={r[\'source_image_id\']:r for r in rows}\n    tasks=[]\n    for source,r in sorted(unique.items()):\n        tasks.append(dict(sample_id=\'clean::\'+source,source_image_id=source,condition=\'clean\',\n                          path=str(Path(locations[\'clean_root\'])/r[\'clean_relpath\']),\n                          image_sha256=r[\'clean_sha256\']))\n    for r in rows:\n        tasks.append(dict(sample_id=\'deg::\'+r[\'variant_id\'],source_image_id=r[\'source_image_id\'],\n                          variant_id=r[\'variant_id\'],condition=\'degraded\',\n                          path=str(Path(locations[\'degraded_root\'])/r[\'degraded_relpath\']),\n                          image_sha256=r[\'degraded_sha256\']))\n    if len({t[\'sample_id\'] for t in tasks})!=len(tasks):raise ValueError(\'Duplicate task IDs\')\n    return tasks\n\n\ndef resume_rows(path):\n    """Repair only an interrupted final JSONL fragment, saving that fragment as evidence."""\n    path=Path(path)\n    if not path.exists():return []\n    data=path.read_bytes();lines=data.splitlines(keepends=True);good=[];offset=0;repaired=False\n    for i,line in enumerate(lines):\n        try:\n            item=json.loads(line)\n        except json.JSONDecodeError:\n            if i!=len(lines)-1:raise ValueError(f\'Corrupted middle of {path}; do not silently drop predictions\')\n            path.with_suffix(path.suffix+\'.interrupted_tail\').write_bytes(line)\n            with path.open(\'r+b\') as f:f.truncate(offset)\n            repaired=True\n            print(\'Recovered interrupted final line:\',path,flush=True)\n            break\n        good.append(item);offset+=len(line)\n    if good and data and not repaired and not data.endswith(b\'\\n\'):\n        with path.open(\'ab\') as f:f.write(b\'\\n\')\n    return good\n\n\ndef worker(a):\n    import math\n    import torch\n    from mmengine.config import Config\n    from mmengine.runner import load_checkpoint\n    from mmengine.utils import import_modules_from_strings\n    from mmocr.apis import TextDetInferencer\n    from mmocr.utils import register_all_modules\n\n    tasks=read_jsonl(a.tasks)\n    out=Path(a.output_dir);out.mkdir(parents=True,exist_ok=True)\n    fingerprint=dict(manifest_sha256=a.manifest_hash,checkpoint_sha256=sha256(a.checkpoint),\n                     config_sha256=sha256(a.config),tasks_sha256=sha256(a.tasks),model=\'monkey\',\n                     worker_source_sha256=sha256(__file__),torch_version=torch.__version__,\n                     gpu_name=torch.cuda.get_device_name(0))\n    if fingerprint[\'checkpoint_sha256\']!=a.checkpoint_hash or fingerprint[\'config_sha256\']!=a.config_hash:\n        raise ValueError(\'Checkpoint/config differ from verified Phase 1 artifacts\')\n    meta=out/\'run_identity.json\'\n    if meta.exists() and json.loads(meta.read_text())!=fingerprint:\n        raise ValueError(\'Existing predictions use a different model/config/manifest/shard. Use a new run directory.\')\n    meta.write_text(json.dumps(fingerprint,indent=2)+\'\\n\')\n    pred_path=out/\'predictions.jsonl\'\n    existing=resume_rows(pred_path)\n    task_ids={t[\'sample_id\'] for t in tasks}\n    seen={r[\'sample_id\'] for r in existing}\n    if len(seen)!=len(existing) or not seen<=task_ids:\n        raise ValueError(\'Duplicate/unexpected resume predictions\')\n    if any(r[\'manifest_sha256\']!=a.manifest_hash or r[\'checkpoint_sha256\']!=a.checkpoint_hash for r in existing):\n        raise ValueError(\'Resume output model/manifest identity mismatch\')\n    todo=[t for t in tasks if t[\'sample_id\'] not in seen]\n    if not todo:\n        print(\'Shard complete; no new inference:\',len(tasks),flush=True)\n        return\n    assert torch.cuda.is_available() and torch.cuda.device_count()==1,\'Worker must see exactly one assigned GPU\'\n    cfg=Config.fromfile(a.config)\n    if cfg.get(\'custom_imports\'):import_modules_from_strings(**cfg.custom_imports)\n    register_all_modules(init_default_scope=True)\n    detector=TextDetInferencer(model=a.config,weights=a.checkpoint,device=\'cuda:0\')\n    detector.model.eval()\n    # Strict-load once more: reject missing random heads rather than accepting a partial checkpoint.\n    load_checkpoint(detector.model,a.checkpoint,map_location=\'cpu\',strict=True)\n    protocol=dict(physical_visible_gpu=os.environ.get(\'CUDA_VISIBLE_DEVICES\'),\n                  gpu_name=torch.cuda.get_device_name(0),torch=torch.__version__,cuda=torch.version.cuda,\n                  model=cfg.model,test_pipeline=cfg.test_dataloader.dataset.pipeline,\n                  effective_inferencer_pipeline=repr(detector.pipeline),\n                  score_policy=\'raw detector output; no extra score filter/sweep\')\n    (out/\'inference_protocol.json\').write_text(json.dumps(protocol,indent=2,default=str)+\'\\n\')\n    def predict(path):\n        with torch.inference_mode():\n            result=detector(path,batch_size=1,return_vis=False,save_vis=False,save_pred=False,show=False)\n        assert len(result[\'predictions\'])==1\n        return result[\'predictions\'][0]\n    # Warmup uses only pixels, never GT, and does not add duplicated saved predictions.\n    for _ in range(2):predict(todo[0][\'path\'])\n    torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()\n    start=time.perf_counter();new_count=0\n    with pred_path.open(\'a\',encoding=\'utf-8\') as f:\n        for task in todo:\n            task_start=time.perf_counter()\n            if sha256(task[\'path\'])!=task[\'image_sha256\']:\n                raise ValueError(f\'Input image bytes changed after audit: {task["sample_id"]}\')\n            torch.cuda.synchronize();t=time.perf_counter()\n            pred=predict(task[\'path\'])\n            torch.cuda.synchronize();elapsed=(time.perf_counter()-t)*1000\n            polygons=[[float(v) for v in p] for p in pred[\'polygons\']]\n            scores=[float(v) for v in pred[\'scores\']]\n            assert len(polygons)==len(scores)\n            assert all(math.isfinite(v) for p in polygons for v in p)\n            assert all(math.isfinite(v) and 0<=v<=1 for v in scores)\n            row={k:v for k,v in task.items() if k not in {\'path\',\'image_sha256\'}}\n            row.update(polygons=polygons,scores=scores,model=\'monkey\',manifest_sha256=a.manifest_hash,\n                       checkpoint_sha256=a.checkpoint_hash,config_sha256=a.config_hash,\n                       inference_ms_with_decode_and_postprocess=elapsed,\n                       processing_ms_with_hash_and_decode=(time.perf_counter()-task_start)*1000)\n            f.write(json_line(row));f.flush();new_count+=1\n            if new_count%25==0 or new_count==len(todo):\n                print(f\'PROGRESS {len(seen)+new_count}/{len(tasks)} | new elapsed {time.perf_counter()-start:.1f}s\',flush=True)\n    (out/\'worker_summary.json\').write_text(json.dumps(dict(new_images=new_count,resumed_images=len(seen),\n         wall_seconds_new=time.perf_counter()-start,max_allocated_gpu_mib=torch.cuda.max_memory_allocated()/1024**2),indent=2)+\'\\n\')\n\n\ndef launch(a):\n    locations=json.loads(Path(a.locations).read_text())\n    tasks=build_tasks(a.manifest,locations,a.smoke_sources)\n    manifest_hash=sha256(a.manifest)\n    root=Path(a.output_dir);root.mkdir(parents=True,exist_ok=True)\n    gpu_ids=[x.strip() for x in a.gpus.split(\',\') if x.strip()]\n    if not gpu_ids or len(gpu_ids)!=len(set(gpu_ids)):raise ValueError(\'Set one or more unique GPU IDs\')\n    processes=[];handles=[];started=time.perf_counter()\n    try:\n        for shard,gpu in enumerate(gpu_ids):\n            subset=tasks[shard::len(gpu_ids)]\n            task_file=root/f\'tasks_gpu{gpu}.jsonl\'\n            content=\'\'.join(json_line(t) for t in subset)\n            if task_file.exists() and task_file.read_text()!=content:\n                raise ValueError(\'Sharding changed; use a new output directory instead of mixing runs\')\n            task_file.write_text(content,encoding=\'utf-8\')\n            env=os.environ.copy();env[\'CUDA_VISIBLE_DEVICES\']=gpu\n            cmd=[sys.executable,str(Path(__file__).resolve()),\'--worker\',\'--tasks\',str(task_file),\n                 \'--output-dir\',str(root/f\'gpu{gpu}\'),\'--manifest-hash\',manifest_hash,\n                 \'--config\',a.config,\'--checkpoint\',a.checkpoint,\n                 \'--config-hash\',a.config_hash,\'--checkpoint-hash\',a.checkpoint_hash]\n            handle=(root/f\'gpu{gpu}.log\').open(\'w\',encoding=\'utf-8\');handles.append(handle)\n            processes.append((gpu,subprocess.Popen(cmd,env=env,stdout=handle,stderr=subprocess.STDOUT)))\n            print(\'Started GPU\',gpu,\'tasks\',len(subset),flush=True)\n        last=0\n        while any(p.poll() is None for _,p in processes):\n            failures=[(g,p.returncode) for g,p in processes if p.poll() not in (None,0)]\n            if failures:raise RuntimeError(f\'Inference worker failed {failures}; see per-GPU logs. Partial outputs are NOT scored.\')\n            if time.perf_counter()-last>=30:\n                last=time.perf_counter()\n                progress={g:sum(1 for _ in (root/f\'gpu{g}/predictions.jsonl\').open(encoding=\'utf-8\'))\n                          if (root/f\'gpu{g}/predictions.jsonl\').exists() else 0 for g,_ in processes}\n                print(\'SHARD PROGRESS\',progress,\'elapsed_s\',round(time.perf_counter()-started),flush=True)\n            time.sleep(1)\n        if any(p.returncode for _,p in processes):raise RuntimeError(\'Worker failed; see logs\')\n    finally:\n        for _,p in processes:\n            if p.poll() is None:p.terminate()\n        for _,p in processes:\n            try:p.wait(timeout=15)\n            except subprocess.TimeoutExpired:\n                p.kill();p.wait(timeout=15)\n        for h in handles:h.close()\n    merged={}\n    for gpu in gpu_ids:\n        for row in read_jsonl(root/f\'gpu{gpu}/predictions.jsonl\'):\n            key=row[\'sample_id\']\n            if key in merged:raise ValueError(\'Duplicate merged prediction: \'+key)\n            merged[key]=row\n    if set(merged)!={t[\'sample_id\'] for t in tasks}:\n        raise ValueError(\'Incomplete or extra merged predictions; not evaluated\')\n    merged_path=root/\'monkey_predictions.jsonl\'\n    merged_path.write_text(\'\'.join(json_line(merged[k]) for k in sorted(merged)),encoding=\'utf-8\')\n    per_gpu={}\n    for gpu in gpu_ids:\n        values=[r[\'processing_ms_with_hash_and_decode\'] for r in read_jsonl(root/f\'gpu{gpu}/predictions.jsonl\')]\n        per_gpu[gpu]=dict(samples=len(values),mean_ms=sum(values)/len(values) if values else 0.)\n    full_count=len(build_tasks(a.manifest,locations))\n    eta=max((v[\'mean_ms\']*(full_count//len(gpu_ids)+1)/3_600_000 for v in per_gpu.values()),default=0.)\n    summary=dict(images=len(tasks),unique_sources=len({t[\'source_image_id\'] for t in tasks}),\n                 manifest_sha256=manifest_hash,predictions_sha256=sha256(merged_path),\n                 per_gpu=per_gpu,wall_seconds_this_launch=time.perf_counter()-started,\n                 estimated_full_inference_hours_excludes_setup_and_evaluation=eta,\n                 note=\'ETA extrapolation from actual inputs, not a guarantee; resume totals include prior images\')\n    (root/\'inference_summary.json\').write_text(json.dumps(summary,indent=2)+\'\\n\')\n    print(json.dumps(summary,indent=2),flush=True)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--worker\',action=\'store_true\')\n    p.add_argument(\'--manifest\');p.add_argument(\'--locations\');p.add_argument(\'--gpus\',default=\'0,1\')\n    p.add_argument(\'--smoke-sources\',type=int,default=0);p.add_argument(\'--output-dir\',required=True)\n    p.add_argument(\'--config\',required=True);p.add_argument(\'--checkpoint\',required=True)\n    p.add_argument(\'--config-hash\',required=True);p.add_argument(\'--checkpoint-hash\',required=True)\n    p.add_argument(\'--tasks\');p.add_argument(\'--manifest-hash\')\n    a=p.parse_args()\n    worker(a) if a.worker else launch(a)\n',encoding="utf-8")
(ART/"scripts"/'visualize_pairs.py').write_text('"""Paired GT/prediction overlays; input images remain unchanged."""\nimport argparse,json\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image,ImageDraw\nfrom shapely.geometry import shape,Polygon\nfrom shared_manifest import overlay\n\n\ndef read(path):\n    return [json.loads(s) for s in Path(path).read_text(encoding=\'utf-8\').splitlines() if s.strip()]\n\n\ndef main(a):\n    rows=read(a.manifest);locations=json.loads(Path(a.locations).read_text())\n    pred={p[\'sample_id\']:p for p in read(a.predictions)}\n    available={r[\'source_image_id\'] for r in rows\n               if \'clean::\'+r[\'source_image_id\'] in pred and \'deg::\'+r[\'variant_id\'] in pred}\n    sources=[]\n    if a.worst and Path(a.worst).is_file():\n        sources=[r[\'source_image_id\'] for r in json.loads(Path(a.worst).read_text())\n                 if r[\'source_image_id\'] in available]\n    sources=list(dict.fromkeys(sources+sorted(available)))[:a.limit]\n    out=Path(a.output_dir);out.mkdir(parents=True,exist_ok=True)\n    cards=[]\n    for row in rows:\n        if row[\'source_image_id\'] not in sources or \'deg::\'+row[\'variant_id\'] not in pred:continue\n        for condition in (\'clean\',\'degraded\'):\n            key=\'clean::\'+row[\'source_image_id\'] if condition==\'clean\' else \'deg::\'+row[\'variant_id\']\n            path=Path(locations[condition+\'_root\'])/row[condition+\'_relpath\']\n            with Image.open(path) as im:\n                im=im.convert(\'RGB\');drawing=ImageDraw.Draw(im)\n                overlay(shape(row[\'roi_\'+condition]),drawing,\'#00BFFF\',3)\n                for item in row[\'gt_\'+condition][\'ignored\']:overlay(shape(item[\'geometry\']),drawing,\'#999999\')\n                for item in row[\'gt_\'+condition][\'valid\']:overlay(shape(item[\'geometry\']),drawing,\'#00FF00\')\n                for coordinates in pred[key][\'polygons\']:\n                    pts=np.asarray(coordinates).reshape(-1,2)\n                    if len(pts)>=3:overlay(Polygon(pts),drawing,\'#FF0000\')\n                im.thumbnail((1280,1280))\n                name=row[\'source_image_id\']+\'_\'+row[\'variant_id\']+\'_\'+condition+\'.jpg\'\n                im.save(out/name,quality=88)\n                cards.append((key,name))\n    import html\n    markup=\'<meta charset="utf-8"><title>Paired detection audit</title><h1>Clean/degraded GT + predictions</h1><p>Green valid GT; gray ignored; cyan valid field of view; red raw predictions.</p>\'\n    markup+=\'\'.join(f\'<figure><figcaption>{html.escape(key)}</figcaption><img style="max-width:100%" src="{html.escape(name)}"></figure>\' for key,name in cards)\n    (out/\'gallery.html\').write_text(markup,encoding=\'utf-8\')\n    print(\'Overlays:\',len(cards),\'in\',out)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    for name in (\'manifest\',\'locations\',\'predictions\',\'output-dir\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--worst\',default=\'\');p.add_argument(\'--limit\',type=int,default=10)\n    main(p.parse_args())\n',encoding="utf-8")
ENV['PYTHONPATH']=str(ART/'scripts')+os.pathsep+str(MMOCR_ROOT)
for p in (ART/'scripts').glob('*.py'):compile(p.read_text(),str(p),'exec')
for name in ('shared_manifest_v1.jsonl','shared_manifest_v1.sha256','evaluation_protocol.json',
             'data_audit.json','data_audit.md','textocr_schema.json','dataset_locations.json','excluded.jsonl','clean_index.csv','degraded_index.csv'):
    target=ART/'shared'/name;target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(SHARED/name,target)
shutil.copytree(SHARED/'gt_overlays',ART/'shared/gt_overlays',dirs_exist_ok=True)
SCRIPTS=ART/'scripts'


In [ ]:
STRICT_SCRIPT=ART/'scripts/strict_checkpoint_check.py'
STRICT_SCRIPT.write_text("""import sys
from mmengine.config import Config
from mmengine.utils import import_modules_from_strings
from mmengine.runner import load_checkpoint
from mmocr.registry import MODELS
from mmocr.utils import register_all_modules
cfg=Config.fromfile(sys.argv[1])
if cfg.get('custom_imports'):import_modules_from_strings(**cfg.custom_imports)
register_all_modules(init_default_scope=True)
model=MODELS.build(cfg.model)
load_checkpoint(model,sys.argv[2],map_location='cpu',strict=True)
print('Phase 2 strict checkpoint check OK')
""")
run([PY,STRICT_SCRIPT,CONFIG,CHECKPOINT],'strict_checkpoint_check.log',cwd=MMOCR_ROOT,env=ENV)
base_command=[PY,SCRIPTS/'run_monkey.py','--manifest',SHARED_RESULT['manifest'],
              '--locations',SHARED_RESULT['locations'],'--gpus',GPU_IDS,
              '--config',CONFIG,'--checkpoint',CHECKPOINT,
              '--config-hash',EXPECTED_CONFIG_SHA256,'--checkpoint-hash',EXPECTED_CHECKPOINT_SHA256]
run(base_command+['--smoke-sources',str(SMOKE_SOURCES),'--output-dir',ART/'smoke'],
    'smoke_inference.log',cwd=MMOCR_ROOT,env=ENV)
smoke_summary=json.loads((ART/'smoke/inference_summary.json').read_text())
print('ACTUAL TEXTVQA SMOKE:',json.dumps(smoke_summary,indent=2))
print('Inference-only ETA (hours; excludes setup/audit/evaluator):',
      smoke_summary['estimated_full_inference_hours_excludes_setup_and_evaluation'])
run([PY,SCRIPTS/'visualize_pairs.py','--manifest',SHARED_RESULT['manifest'],
     '--locations',SHARED_RESULT['locations'],'--predictions',ART/'smoke/monkey_predictions.jsonl',
     '--output-dir',ART/'smoke/overlays','--limit','4'],'smoke_overlays.log',env=ENV)


## Full inference — tự tiếp tục sau audit và smoke đạt
Mặc định FULL chạy toàn bộ giao hợp lệ trong cùng lượt Run All; không có gate chép hash thủ công.
Không sửa resize/threshold/degradation sau khi xem kết quả. Mọi prediction được ghi sau từng ảnh,
resume chỉ chấp nhận cùng checkpoint/config/manifest/shard fingerprint. Không chấm output thiếu.


In [ ]:
if RUN_STAGE=='FULL':
    assert file_sha256(SHARED_RESULT['manifest'])==data_identity['manifest_sha256'],'Manifest changed after audit.'
    assert file_sha256(SHARED_RESULT['protocol'])==data_identity['protocol_sha256'],'Protocol changed after audit.'
    run(base_command+['--output-dir',ART/'full'],'full_inference.log',cwd=MMOCR_ROOT,env=ENV)
    run([PY,SCRIPTS/'evaluate_shared.py','--manifest',SHARED_RESULT['manifest'],
         '--protocol',SHARED_RESULT['protocol'],'--predictions',ART/'full/monkey_predictions.jsonl',
         '--output-dir',ART/'metrics','--bootstrap',str(BOOTSTRAP_REPETITIONS)],'evaluation.log',env=ENV)
    run([PY,SCRIPTS/'visualize_pairs.py','--manifest',SHARED_RESULT['manifest'],
         '--locations',SHARED_RESULT['locations'],'--predictions',ART/'full/monkey_predictions.jsonl',
         '--worst',ART/'metrics/worst_recall_sources.json','--output-dir',ART/'visual_examples',
         '--limit','12'],'paired_error_overlays.log',env=ENV)
    print((ART/'metrics/clean_vs_degraded.md').read_text())
else:
    print('AUDIT_SMOKE complete theo lựa chọn cấu hình. Chọn FULL để chạy toàn bộ; không cần điền hash thủ công.')


## Lưu artifact để bàn giao
ZIP chỉ chứa audit, manifest, scripts, predictions, metrics và overlays; không chứa env/source repo/data ảnh gốc.
Nếu Kaggle session bị ngắt, giữ output đã có và logs. Venv copy sang mount root khác cần dựng lại;
prediction resume còn yêu cầu dữ liệu và model fingerprints giống nhau, không nối các run khác nhau.

Baseline hiện chấm toàn bộ giao hợp lệ để khảo sát. Trước fine-tune sau này, phải chốt split theo source ID;
không dùng ảnh đã train/tune làm final test. TextOCR train xuất hiện trong nguồn pretraining Monkey,
nên không gọi tập này là unseen/zero-shot benchmark hay dữ liệu suy thoái chụp thật.


In [ ]:
for p in sorted((ART/('visual_examples' if RUN_STAGE=='FULL' else 'smoke/overlays')).glob('*.jpg'))[:4]:
    print(p.name);display(DisplayImage(filename=str(p),width=800))
inventory={str(p.relative_to(ART)):dict(bytes=p.stat().st_size,sha256=file_sha256(p))
           for p in ART.rglob('*') if p.is_file() and p.name!='inventory_sha256.json'}
save_json(ART/'inventory_sha256.json',inventory)
zip_path=shutil.make_archive(str(ROOT/('phase2_monkey_full' if RUN_STAGE=='FULL' else 'phase2_audit_smoke')),'zip',ART)
from IPython.display import FileLink
print('DOWNLOAD:',zip_path);display(FileLink(zip_path))
print('Paddle must reuse the exact shared manifest/protocol/evaluator bytes and verify SHA256.')
